# Imports and Data Prep

In [ ]:
import os
import sys
import gc
import pandas as pd
import joblib
import matplotlib.pyplot as plt
import nibabel as nib
import numpy as np

from sklearn.decomposition import FactorAnalysis
from sklearn.cluster import AgglomerativeClustering
from tqdm import tqdm
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor
from sklearn.decomposition import TruncatedSVD
from scipy.stats import zscore
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

import importlib
from collections import defaultdict
import traceback
from matplotlib.backends.backend_pdf import PdfPages
from typing import Union
from joblib import Parallel, delayed
from multiprocessing import Pool, cpu_count

from scipy.ndimage import percentile_filter, gaussian_filter
from scipy import stats
from scipy.cluster.hierarchy import linkage, dendrogram, fcluster

from sklearn.metrics.pairwise import cosine_similarity
from scipy.signal import fftconvolve
import pickle
import tifffile
from mpl_toolkits.axes_grid1.inset_locator import inset_axes


In [ ]:
BASE_DIR = Path("/mnt/storage-raid10/Yun/analysis_output/chemogenetic")
PROJ_ID = "hcrt-trpv1_huc-h2b-g8m_csn_120min"
PROJ_CTRL = "huc-h2b-g8m_csn_120min"
EXPT_FISH_LIST = [
    "251008_hcrt-trpv1_huc-h2b-g8m_csn_10uM_fish4",
    "251102_hcrt-trpv1_huc-h2b-g8m_csn_10uM_fish1",
    "251102_hcrt-trpv1_huc-h2b-g8m_csn_10uM_fish2",
    "251210_hcrt-trpv1_huc-h2b-g8m_csn_10uM_fish1",
    "251210_hcrt-trpv1_huc-h2b-g8m_csn_10uM_fish2", 
    "251210_hcrt-trpv1_huc-h2b-g8m_csn_10uM_fish3",
    "260514_hcrt-trpv1_huc-h2b-g8m_csn_10uM_fish1",
    "260514_hcrt-trpv1_huc-h2b-g8m_csn_10uM_fish2",
    "260515_hcrt-trpv1_huc-h2b-g8m_csn_10uM_fish1"
]

N_COMPONENTS = 15
N_CLUSTERS = 18
PHASIC_DPRIME_THRESH = 0.25
RESPONSE_TYPES = ["tonic_pos", "tonic_neg", "phasic_pos", "phasic_neg"]

DIR_ANTS_OUTPUT = str(BASE_DIR)

example_fish = '251008_hcrt-trpv1_huc-h2b-g8m_csn_10uM_fish4' #used for testing

In [ ]:
#Parameters 

# IMAGING SPECS 
sec_per_volume = 1
volume_per_sec = 1
n_slices = 40
depth = 250
binning = 1
res_x = 1.52*binning
res_y = 1.52*binning
res_z = depth/n_slices
rotation_k = 2 

# DRUG PERFUSION PARAMETER 
drug_uM   = 10.0
V_ml     = 15.0
Q_ml_min = 4.5


baseline_start = 0 * 60 * volume_per_sec
baseline_end = 45 * 60 * volume_per_sec

# define drug perfusion start & end time frame 
drug_start = 46  * 60 * volume_per_sec
drug_end = 90  * 60 * volume_per_sec

# define E3+DMSO wash out start & end time frame
wash_start = 91 * 60 * volume_per_sec
wash_end = 120 * 60 * volume_per_sec


#  define delta F / F 's baseline percentile
df_f_percentile = 20

# define F_tonic's window size and percentile
f_tonic_window_size = 600  #seconds
f_tonic_percentile = 20

# permutation test parameters
p_thresh_permutation = 0.005
n_resample_permutation = 500

# RUN BH-FDR FILTERING CODE
BH_Q = 0.05  



input_tag    = "C"
K_global     = 600
drift_global = 1       # Order 1 (Linear)
lam_global   = 0.5
lag_global   = 0  

param_folder_name = f"in{input_tag}_K{K_global}_drift{drift_global}_lam{lam_global}_lag{lag_global}"

CLIP_ABS_DZ = 50.0 
INCLUDED_BASELINE = 15.0     # minutes of baseline to use (matches GLM fit_baseline_sec)
NULL_TAG = "iaaft"
RESPONDER_NULL_THRESH = 95  # threshold for null
L_MIN = 20.0   # plateau duration in minutes



In [ ]:
#low mem method to load fish data and responders

def process_fish(fish_id):
    fish_dir = BASE_DIR / PROJ_ID / fish_id
    
    try:
        f_tonic = np.load(fish_dir / "f_tonic.npy", mmap_mode="r")
        f_phasic = np.load(fish_dir / "f_phasic.npy", mmap_mode="r")
        
      
        tonic_pos_idx = np.load(fish_dir / "tonic_pos_glm_iaaft_nullp95_idxs.npy")
        tonic_neg_idx = np.load(fish_dir / "tonic_neg_glm_iaaft_nullp95_idxs.npy")
        dprime = np.load(fish_dir / "phasic_dprime_cells_raw.npy")

      
        all_tonic_idx = np.union1d(tonic_pos_idx, tonic_neg_idx)
        all_phasic_idx = np.where(np.abs(dprime) >= PHASIC_DPRIME_THRESH)[0]

     
        tonic_slice = np.array(f_tonic[all_tonic_idx, :])
        phasic_slice = np.array(f_phasic[all_phasic_idx, :])

        return fish_id, tonic_slice, phasic_slice

    except FileNotFoundError:
        return None

# Main Execution
tonic_data = {}
phasic_data = {}

print("Started processing fish folders")


with ThreadPoolExecutor(max_workers=3) as executor:
    results = executor.map(process_fish, EXPT_FISH_LIST)

for result in results:
    if result is not None:
        fish_id, t_res, p_res = result
        tonic_data[fish_id] = t_res
        phasic_data[fish_id] = p_res

print(f"Completed processing for {len(tonic_data)} fish.")


In [ ]:
'''

#For later with control

ctrl_ID_1 = ''
ctrl_ID_2 = ''
ctrl_ID_3 = ''
ctrl_ID_4 = ''
ctrl_ID_5 = ''
ctrl_ID_6 = ''
ctrl_ID_7 = ''

#ctrl_expt_ID_list = [ctrl_ID_1, ctrl_ID_2, ctrl_ID_3, ctrl_ID_4, ctrl_ID_5, ctrl_ID_6, ctrl_ID_7]

expt_ID_list = ctrl_expt_ID_list + expt_fish_list

'''

In [ ]:
from scipy.stats import zscore

z_phasic = {}

for fish_id in EXPT_FISH_LIST:
  
    if fish_id in phasic_data:
    
        raw_phasic_traces = phasic_data[fish_id]

        z_phasic_traces = zscore(raw_phasic_traces, axis=1)

        z_phasic[fish_id] = z_phasic_traces

    else:
        print(f"Fish data matrix not found in memory for: {fish_id}\n")

print("All experimental fish traces have been successfully normalized!")


# Hyperparameter Optimization


## Scree Plot for determining optimal number of factors

In [ ]:
#Scree Plot for determining optimal number of factors
def plot_scree_curve(raw_phasic_traces, target_fish):

    try:
        clean_raw_traces = raw_phasic_traces[np.var(raw_phasic_traces, axis=1) > 0]
        n_dropped = len(raw_phasic_traces) - len(clean_raw_traces)

        if n_dropped > 0:
            print(f"Automatically filtered out {n_dropped} dead/flat cells with zero variance.")

        z_phasic_traces = zscore(clean_raw_traces, axis=1)
        data_matrix = z_phasic_traces.T
        print(f"Evaluating factor spectrum for clean matrix shape: {data_matrix.shape}...")

        n_factors = 30 #max number of factor it will test
        svd = TruncatedSVD(n_components=n_factors, random_state=42)
        svd.fit(data_matrix)

        plt.figure(figsize=(10, 5))
        plt.plot(range(1, n_factors + 1), svd.explained_variance_ratio_, 'o-', linewidth=2, color='#1f77b4')
        plt.axvline(x=20, color='r', linestyle='--', label="Baseline # of factors (K=20)")
        plt.title(f"Scree for {target_fish}", fontsize=14)
        plt.xlabel("Number of Factors", fontsize=12)
        plt.ylabel("Explained Variance Ratio", fontsize=12)
        plt.xticks(range(1, n_factors + 1))
        plt.legend(loc="upper right")
        plt.grid(True, alpha=0.3)
        plt.show()
        
    except Exception as e:
        print(f"Error occurred while plotting scree curve for {target_fish}: {e}")


print("Started processing fish folders")

for fish_id in EXPT_FISH_LIST:
    if fish_id in phasic_data:
        raw_phasic_traces = phasic_data[fish_id]
        plot_scree_curve(raw_phasic_traces, fish_id)
    else:
        print(f"Fish data matrix not found in memory for: {fish_id}\n")

print(f"Completed processing for {len(tonic_data)} fish.")


As shown in the elbow plots for the 9 expt fish, the k (number of factors ) where the EVR was at a solid point and adding more clusters yielded diminishing returns was on average 5. 

## Elbow Plot for determining optimal number of clusters

In [ ]:
#Elbow Plot for determining optimal number of clusters
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import zscore
from sklearn.cluster import KMeans

def plot_cluster_elbow_curve(raw_phasic_traces, target_fish):
    try:
        clean_raw_traces = raw_phasic_traces[np.var(raw_phasic_traces, axis=1) > 0]
        n_dropped = len(raw_phasic_traces) - len(clean_raw_traces)
        if n_dropped > 0:
            print(f"Automatically filtered out {n_dropped} dead/flat cells with zero variance.")
            
        z_phasic_traces = zscore(clean_raw_traces, axis=1)
        
        # KMEANS clusters rows, rows are cells, so we are good!
    
        data_matrix = z_phasic_traces
        print(f"Evaluating clusters for matrix shape: {data_matrix.shape}...")
        
        max_clusters = 15  #number of max clusters to test for the elbow method
        wcss = []          
        
        
        cluster_range = range(1, max_clusters + 1)
        for k in cluster_range:
            kmeans = KMeans(n_clusters=k, random_state=42, n_init=5)
            kmeans.fit(data_matrix)
            wcss.append(kmeans.inertia_) # inertia_ is the WCSS
            
      
        plt.figure(figsize=(10, 5))
        plt.plot(cluster_range, wcss, 'o-', linewidth=2, color='blue')
        plt.title(f"K-Means Elbow Plot for {target_fish}", fontsize=14)
        plt.xlabel("Number of Clusters (K)", fontsize=12)
        plt.ylabel("Within-Cluster Sum of Squares (Inertia)", fontsize=12)
        plt.xticks(cluster_range)
        plt.grid(True, alpha=0.3)
        plt.show()
        
    except Exception as e:
        print(f"Error occurred while plotting elbow curve for {target_fish}: {e}")

print("Started processing fish folders")
for fish_id in EXPT_FISH_LIST:
    if fish_id in phasic_data:
        raw_phasic_traces = phasic_data[fish_id]
        plot_cluster_elbow_curve(raw_phasic_traces, fish_id)
    else:
        print(f"Fish data matrix not found in for: {fish_id}\n")


## Silhouette scores and plots for verifying number of clusters

In [ ]:
def plot_silhouette_curve(raw_phasic_traces, target_fish):
    try:
  
        clean_raw_traces = raw_phasic_traces[np.var(raw_phasic_traces, axis=1) > 0]
        z_phasic_traces = zscore(clean_raw_traces, axis=1) 
        
        # 2. Factor Analysis 
        n_optimal_factors = 15
        fa = FactorAnalysis(n_components=n_optimal_factors, random_state=42)
        # data_matrix will correctly be shape: (n_cells, n_optimal_factors)
        data_matrix = fa.fit_transform(z_phasic_traces)
        
        print(f"Evaluating cell silhouette profile across factors for matrix: {data_matrix.shape}")
        
        min_clusters = 2
        max_clusters = 30
        silhouette_scores = []
        cluster_range = list(range(min_clusters, max_clusters + 1))
        
        for k in cluster_range:
            kmeans = KMeans(n_clusters=k, random_state=42, n_init=5)
            cluster_labels = kmeans.fit_predict(data_matrix)
            
            score = silhouette_score(data_matrix, cluster_labels, sample_size=10000, random_state=42)
            silhouette_scores.append(score)
            
        if silhouette_scores:
            best_idx = int(np.argmax(silhouette_scores))
            best_k = cluster_range[best_idx]
            best_score = silhouette_scores[best_idx]
            print(f'Highest silhouette score is for {best_k} clusters with a score of {best_score:.3f}')
            
            # 4. Plotting
            plt.figure(figsize=(10, 5))
            plt.plot(cluster_range, silhouette_scores, 'o-', linewidth=2, color='#9467bd')
            plt.title(f"Silhouette Profile for {target_fish} Cells (Factor Space)", fontsize=14)
            plt.xlabel("Number of Clusters (K)", fontsize=12)
            plt.ylabel("Average Silhouette Score", fontsize=12)
            plt.xticks(cluster_range)
            plt.grid(True, alpha=0.3)
            plt.show()
            
            return best_k, data_matrix
            
    except Exception as e:
        print(f"Error occurred while plotting silhouette curve for {target_fish}: {e}")
        return None, None

# Loop
print("Started processing fish folders")
for fish_id in EXPT_FISH_LIST:
    if fish_id in phasic_data:
        plot_silhouette_curve(phasic_data[fish_id], fish_id)
    else:
        print(f"Fish data matrix not found for: {fish_id}\n")

In [ ]:
# ...existing code...
def find_silhouette_scores(raw_phasic_traces, target_fish):
    clean_raw_traces = raw_phasic_traces[np.var(raw_phasic_traces, axis=1) > 0]
    z_phasic_traces = zscore(clean_raw_traces, axis=1) 

    n_optimal_factors = 15
    fa = FactorAnalysis(n_components=n_optimal_factors, random_state=42)
    data_matrix = fa.fit_transform(z_phasic_traces)

    print(f"Evaluating cell silhouette profile across factors for matrix: {data_matrix.shape}")

    min_clusters = 2
    max_clusters = 30
    cluster_range = list(range(min_clusters, max_clusters + 1))
    silhouette_scores = []
    for n_clusters in cluster_range:
        clusterer = AgglomerativeClustering(n_clusters=n_clusters, linkage='ward')
        cluster_labels = clusterer.fit_predict(data_matrix)

        clf = NearestCentroid()
        clf.fit(data_matrix, cluster_labels)
        print("Centroids:")
        print(clf.centroids_)

        # safe sample size
        n_samples = data_matrix.shape[0]
        sample_size = min(50000, n_samples)
        if sample_size < n_samples:
            score = silhouette_score(data_matrix, cluster_labels, sample_size=sample_size, random_state=42)
        else:
            score = silhouette_score(data_matrix, cluster_labels)
        silhouette_scores.append(score)

    # after collecting all scores
    best_idx = int(np.argmax(silhouette_scores))
    best_k = cluster_range[best_idx]
    best_score = silhouette_scores[best_idx]
    print(f'Highest silhouette score is for {best_k} clusters with a score of {best_score:.3f}')

    plt.figure(figsize=(10, 5))
    plt.plot(cluster_range, silhouette_scores, 'o-', linewidth=2, color='#9467bd')
    plt.title(f"Silhouette Profile for {target_fish} Cells (Factor Space)", fontsize=14)
    plt.xlabel("Number of Clusters (K)", fontsize=12)
    plt.ylabel("Average Silhouette Score", fontsize=12)
    plt.xticks(cluster_range)
    plt.grid(True, alpha=0.3)
    plt.show()

    return best_k, data_matrix
# ...existing code...

In [ ]:
import pandas as pd
import numpy as np
from sklearn.cluster import AgglomerativeClustering
from sklearn.metrics import silhouette_samples, silhouette_score
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from mpl_toolkits.mplot3d import Axes3D
from sklearn.neighbors import NearestCentroid

def find_silhouette_scores(raw_phasic_traces, target_fish):
    clean_raw_traces = raw_phasic_traces[np.var(raw_phasic_traces, axis=1) > 0]
    z_phasic_traces = zscore(clean_raw_traces, axis=1) 

    n_optimal_factors = 15
    fa = FactorAnalysis(n_components=n_optimal_factors, random_state=42)
    data_matrix = fa.fit_transform(z_phasic_traces)

    print(f"Evaluating cell silhouette profile across factors for matrix: {data_matrix.shape}")

    min_clusters = 2
    max_clusters = 30
    cluster_range = list(range(min_clusters, max_clusters + 1))
    silhouette_scores = []

    # Compute the silhouette scores for each sample

    for n_clusters in cluster_range:
        clusterer = AgglomerativeClustering(n_clusters=n_clusters, linkage='ward')
        cluster_labels = clusterer.fit_predict(data_matrix)

        # safe sample size
        n_samples = data_matrix.shape[0]
        sample_size = min(50000, n_samples)
        if sample_size < n_samples:
            score = silhouette_score(data_matrix, cluster_labels, sample_size=sample_size, random_state=42)
        else:
            score = silhouette_score(data_matrix, cluster_labels)
        silhouette_scores.append(score)

    # after collecting all scores
    best_idx = int(np.argmax(silhouette_scores))
    best_k = cluster_range[best_idx]
    best_score = silhouette_scores[best_idx]
    print(f'Highest silhouette score is for {best_k} clusters with a score of {best_score:.3f}')

    plt.figure(figsize=(10, 5))
    plt.plot(cluster_range, silhouette_scores, 'o-', linewidth=2, color='#9467bd')
    plt.title(f"Silhouette Profile for {target_fish} Cells (Factor Space)", fontsize=14)
    plt.xlabel("Number of Clusters (K)", fontsize=12)
    plt.ylabel("Average Silhouette Score", fontsize=12)
    plt.xticks(cluster_range)
    plt.grid(True, alpha=0.3)
    plt.show()

    return best_k, data_matrix

print("Started processing fish folders")
for fish_id in EXPT_FISH_LIST:
    if fish_id in phasic_data:
        find_silhouette_scores(phasic_data[fish_id], fish_id)
    else:
        print(f"Fish data matrix not found for: {fish_id}\n")



After calculating the average among the 9 best n_clusters, the best n_clusters was about 20. However, because one fish had a very low score for 20, 18 was chosen, as it achieved fair results across all of the fish.

## Summary:

Optimal number of factors: 5 |
Optimal number of clusters: 7

# Step 1: Pipeline for the FA and Clustering 

In [ ]:
import sys
import pandas as pd
import joblib
from tqdm import tqdm
from pathlib import Path
from sklearn.decomposition import TruncatedSVD

import os
import gc
from pathlib import Path
import numpy as np
import joblib
from scipy.stats import zscore
import matplotlib.pyplot as plt
import nibabel as nib
from sklearn.decomposition import FactorAnalysis
from sklearn.cluster import AgglomerativeClustering
from concurrent.futures import ProcessPoolExecutor 
import ants
from sklearn.decomposition import FactorAnalysis

from concurrent.futures import ThreadPoolExecutor
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
#Parameters 

# IMAGING SPECS 
sec_per_volume = 1
volume_per_sec = 1
n_slices = 40
depth = 250
binning = 1
res_x = 1.52*binning
res_y = 1.52*binning
res_z = depth/n_slices
rotation_k = 2 

# DRUG PERFUSION PARAMETER 
drug_uM   = 10.0
V_ml     = 15.0
Q_ml_min = 4.5


baseline_start = 0 * 60 * volume_per_sec
baseline_end = 45 * 60 * volume_per_sec

# define drug perfusion start & end time frame 
drug_start = 46  * 60 * volume_per_sec
drug_end = 90  * 60 * volume_per_sec

# define E3+DMSO wash out start & end time frame
wash_start = 91 * 60 * volume_per_sec
wash_end = 120 * 60 * volume_per_sec


#  define delta F / F 's baseline percentile
df_f_percentile = 20

# define F_tonic's window size and percentile
f_tonic_window_size = 600  #seconds
f_tonic_percentile = 20

# permutation test parameters
p_thresh_permutation = 0.005
n_resample_permutation = 500

# RUN BH-FDR FILTERING CODE
BH_Q = 0.05  



input_tag    = "C"
K_global     = 600
drift_global = 1       # Order 1 (Linear)
lam_global   = 0.5
lag_global   = 0  

param_folder_name = f"in{input_tag}_K{K_global}_drift{drift_global}_lam{lam_global}_lag{lag_global}"

CLIP_ABS_DZ = 50.0 
INCLUDED_BASELINE = 15.0     # minutes of baseline to use (matches GLM fit_baseline_sec)
NULL_TAG = "iaaft"
RESPONDER_NULL_THRESH = 95  # threshold for null
L_MIN = 20.0   # plateau duration in minutes


BASE_DIR = Path("/mnt/storage-raid10/Yun/analysis_output/chemogenetic")
PROJ_ID = "hcrt-trpv1_huc-h2b-g8m_csn_120min"
PROJ_CTRL = "huc-h2b-g8m_csn_120min"

PERCENTILE_CUTOFF = 90

EXPT_FISH_LIST = [
    "251008_hcrt-trpv1_huc-h2b-g8m_csn_10uM_fish4",
    "251102_hcrt-trpv1_huc-h2b-g8m_csn_10uM_fish1",
    "251102_hcrt-trpv1_huc-h2b-g8m_csn_10uM_fish2",
    "251210_hcrt-trpv1_huc-h2b-g8m_csn_10uM_fish1",
    "251210_hcrt-trpv1_huc-h2b-g8m_csn_10uM_fish2", 
    "251210_hcrt-trpv1_huc-h2b-g8m_csn_10uM_fish3",
    "260514_hcrt-trpv1_huc-h2b-g8m_csn_10uM_fish1",
    "260514_hcrt-trpv1_huc-h2b-g8m_csn_10uM_fish2",
    "260515_hcrt-trpv1_huc-h2b-g8m_csn_10uM_fish1"
]
RESPONSE_TYPES = ["tonic_pos",
                   "tonic_neg",
                    "phasic_pos",
                    "phasic_neg"
                 ]

DIR_ANTS_OUTPUT = str(BASE_DIR)

example_fish = '251008_hcrt-trpv1_huc-h2b-g8m_csn_10uM_fish4' #used for testing

N_COMPONENTS = 15
N_CLUSTERS = 7


#PHASIC_DPRIME_THRESH = 0.25
EXPT_FISH_LIST = [
    "251008_hcrt-trpv1_huc-h2b-g8m_csn_10uM_fish4",
    "251102_hcrt-trpv1_huc-h2b-g8m_csn_10uM_fish1",
    "251102_hcrt-trpv1_huc-h2b-g8m_csn_10uM_fish2",
]

In [ ]:
from multiprocessing import Pool, cpu_count

def process_single_experiment(args):
    expt_ID, PERCENTILE_CUTOFF, PROJ_ID, DIR_ANTS_OUTPUT, medoid_types, stim_start, stim_end, use_qc, t, save_dendrogram, max_cells, n_components = args
    
    dir_expt = os.path.join(DIR_ANTS_OUTPUT, PROJ_ID, expt_ID)
    OUTPUT_DIR = os.path.join(dir_expt, f"FA_agglo_clustering_f={n_components}_c={t}_t={t}")
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    
    loaded_traces = {}

    for label in medoid_types:
        is_tonic = "tonic" in label
        data_filename = "f_tonic.npy" if is_tonic else "f_phasic.npy"
        data_path = os.path.join(dir_expt, data_filename)
        
        if not os.path.exists(data_path):
            continue

        if data_filename not in loaded_traces:
            loaded_traces[data_filename] = np.load(data_path)
        traces = loaded_traces[data_filename]

        
        if is_tonic:
            tonic_score = np.var(traces[:, stim_start:stim_end], axis=1)
            
            if 'pos' in label:
                tonic_cutoff = np.percentile(tonic_score, PERCENTILE_CUTOFF)
                idxs = np.where(tonic_score >= tonic_cutoff)[0]
            else:
                tonic_cutoff = np.percentile(tonic_score, 100 - PERCENTILE_CUTOFF)
                idxs = np.where(tonic_score <= tonic_cutoff)[0]
            
        else:
            dprime_path = os.path.join(dir_expt, "phasic_dprime_cells_raw.npy")
            if not os.path.exists(dprime_path):
                continue
            dprime = np.load(dprime_path)
            
            if "pos" in label:
                top_percentile_cutoff = np.percentile(dprime, PERCENTILE_CUTOFF)
                idxs = np.where(dprime >= top_percentile_cutoff)[0]
            else:
                top_percentile_cutoff = np.percentile(dprime, 100 - PERCENTILE_CUTOFF)
                idxs = np.where(dprime <= top_percentile_cutoff)[0]

        idxs = idxs[idxs < traces.shape[0]]
        if len(idxs) == 0:
            continue
            
        trace_subset = traces[idxs, stim_start:stim_end]
        z_traces = zscore(trace_subset, axis=1)
        valid_mask = np.all(np.isfinite(z_traces), axis=1)
        
        if np.sum(valid_mask) < 4:
            continue
            
        z_traces = z_traces[valid_mask]
        idxs = idxs[valid_mask]

        MAX_TARGET_CELLS = 50000
        if len(idxs) > MAX_TARGET_CELLS:
    
            metric_arr = tonic_score[idxs] if is_tonic else dprime[idxs]
            
            if "pos" in label:
                top_orders = np.argsort(metric_arr)[-MAX_TARGET_CELLS:]
            else:
                top_orders = np.argsort(metric_arr)[:MAX_TARGET_CELLS]
        
            z_traces = z_traces[top_orders]
            idxs = idxs[top_orders]



        print(f"Proceeding to FA + clustering with {z_traces.shape[0]} cells")
        try:
            fa = FactorAnalysis(n_components=n_components, random_state=0)
            latent = fa.fit_transform(z_traces)
            fa_model_path = os.path.join(OUTPUT_DIR, f"FA_model_{label}.joblib")
            joblib.dump(fa, fa_model_path)
            
            del z_traces, latent, trace_subset, idxs
        except Exception:
            continue
        
    del loaded_traces
    gc.collect()
    return expt_ID

def hierarchical_clustering_parallel(
    EXPT_FISH_LIST, PERCENTILE_CUTOFF, PROJ_ID, DIR_ANTS_OUTPUT, RESPONSE_TYPES, stim_start, stim_end, use_qc, t, save_dendrogram, max_cells, n_components
):
    tasks = [(
        expt_ID, PERCENTILE_CUTOFF, PROJ_ID, DIR_ANTS_OUTPUT, RESPONSE_TYPES, stim_start, stim_end, use_qc, t, save_dendrogram, max_cells, n_components
    ) for expt_ID in EXPT_FISH_LIST]
    
    num_workers = max(1, int(cpu_count() * 0.75)) 
    
    print(f"Starting parallel execution with {num_workers} workers.")
    
    # progress bar
    with Pool(processes=num_workers) as pool:
        results = list(tqdm(pool.imap_unordered(process_single_experiment, tasks), total=len(tasks), desc="Parallel clustering"))
        
    return results

if __name__ == "__main__":
    PERCENTILE_CUTOFF = 90
    hierarchical_clustering_parallel(
        EXPT_FISH_LIST = EXPT_FISH_LIST,
        PERCENTILE_CUTOFF = 90,
        PROJ_ID = PROJ_ID,
        DIR_ANTS_OUTPUT = DIR_ANTS_OUTPUT,
        RESPONSE_TYPES = RESPONSE_TYPES,
        stim_start = 2700,
        stim_end = 7200,
        use_qc = False,
        t = N_CLUSTERS,
        save_dendrogram = True,
        max_cells = 5000,
        n_components = N_COMPONENTS
    )


In [ ]:
# RELATIVE THRESHOLD USE
import gc
from concurrent.futures import ThreadPoolExecutor
import os
from pathlib import Path
import numpy as np
from scipy.stats import zscore

BASE_DIR = Path("/mnt/storage-raid10/Yun/analysis_output/chemogenetic")
PROJ_ID = "hcrt-trpv1_huc-h2b-g8m_csn_120min"

N_COMPONENTS = 15
N_CLUSTERS = 7
RESPONSE_TYPES = ["tonic_pos", "tonic_neg", "phasic_pos", "phasic_neg"]
stim_start_post = 2700
stim_end_post = 7200
RELATIVE_PERCENTILE = 0.95

loaded_cluster_data = {
    fish: {cat: {} for cat in RESPONSE_TYPES} for fish in EXPT_FISH_LIST
}


def process_fish_analysis(fish_id):
    fish_dir = BASE_DIR / PROJ_ID / fish_id
    cluster_dir = (
        fish_dir
        /  f'FA_agglo_clustering_f={N_COMPONENTS}_c={N_CLUSTERS}_t={N_CLUSTERS}'
    )

    if not cluster_dir.exists():
        print(f"Skipping {fish_id} — Path not found: {cluster_dir.name}")
        return None

    fish_results = {cat: {} for cat in RESPONSE_TYPES}

    for category in RESPONSE_TYPES:
        is_tonic = "tonic" in category
        data_filename = "f_tonic.npy" if is_tonic else "f_phasic.npy"
        data_path = fish_dir / data_filename

        if not data_path.exists():
            print(f"data path does not exist for {data_path}")
            continue

        try:
            all_cluster_cells = []
            cluster_cell_mappings = {}

            for c_idx in range(1, N_CLUSTERS + 1):
                idx_file = cluster_dir / f"{category}_c{c_idx}_idxs.npy"
                if idx_file.exists():
                    c_cells = np.load(idx_file)
                    if len(c_cells) > 0:
                        all_cluster_cells.append(c_cells)
                        cluster_cell_mappings[c_idx] = c_cells
                else:
                    print(f"idx file for {idx_file} does not exist")

            if not all_cluster_cells:
                continue

            total_responders = np.concatenate(all_cluster_cells)
            unique_responders, inverse_mapping = np.unique(
                total_responders, return_inverse=True
            )

            raw_traces_mmap = np.load(data_path, mmap_mode="r")

            pooled_post_traces = np.array(
                raw_traces_mmap[unique_responders, stim_start_post:stim_end_post]
            )
            pooled_full_traces = np.array(
                raw_traces_mmap[unique_responders, 0:7200]
            )


            if "pos" in category:
                cell_intensities = np.mean(pooled_post_traces, axis=1)
            else: 
                cell_intensities = np.abs(np.mean(pooled_post_traces, axis=1))

            fish_responder_z = zscore(cell_intensities)

            
            if np.any(np.isfinite(fish_responder_z)):
                cutoff_threshold = np.nanpercentile(
                    fish_responder_z, RELATIVE_PERCENTILE * 100
                )
                relative_top_5_mask = fish_responder_z >= cutoff_threshold
            else:
                relative_top_5_mask = np.zeros_like(
                    unique_responders, dtype=bool
                )
    

            current_idx_pointer = 0
            for c_idx in range(1, N_CLUSTERS + 1):
                if c_idx not in cluster_cell_mappings:
                    continue

                n_cells_in_cluster = len(cluster_cell_mappings[c_idx])
                cluster_inverse_subset = inverse_mapping[
                    current_idx_pointer : current_idx_pointer
                    + n_cells_in_cluster
                ]
                current_idx_pointer += n_cells_in_cluster

                c_post = pooled_post_traces[cluster_inverse_subset]
                c_full = pooled_full_traces[cluster_inverse_subset]

        
                c_relative_mask = relative_top_5_mask[cluster_inverse_subset]

                z_post = zscore(c_post, axis=1)
                z_full = zscore(c_full, axis=1)

                valid_mask = (
                    np.all(np.isfinite(z_post), axis=1)
                    & np.all(np.isfinite(z_full), axis=1)
                    & c_relative_mask
                )

                if np.sum(valid_mask) > 0:
                    fish_results[category][c_idx] = {
                        "post_drug": z_post[valid_mask],
                        "full_timeline": z_full[valid_mask],
                    }

        except Exception as e:
            print(f"Error loading {category} for {fish_id}: {e}")
            continue

    return fish_id, fish_results


print("Loading Dual-Window post-drug and full-timeline matrices from disk...")
with ThreadPoolExecutor(max_workers=3) as executor:
    results = executor.map(process_fish_analysis, EXPT_FISH_LIST)

for result in results:
    if result is not None:
        fish_id, fish_results = result
        loaded_cluster_data[fish_id] = fish_results

print("Data loading completed successfully!")

In [ ]:
import nibabel as nib

TEMPLATE_BRAIN_PATH = "/mnt/storage-raid10/Yun/analysis_output/registration/template_mean_brain.nii.gz"
SUMMARY_PLOTS_DIR = "/ssd-pool/james/lightsheet/Zebrafish-whole-brain-analysis/unsupervised_plots_summary"

CATEGORY_COLORS = {
    'tonic_pos': 'orange',
    'tonic_neg': 'blue',
    'phasic_pos': 'red',    
    'phasic_neg': 'purple'
}

LABEL_TITLES = { 
    'tonic_pos': 'Tonic(+)', 
    'tonic_neg': 'Tonic(-)', 
    'phasic_pos': 'Phasic(+)', 
    'phasic_neg': 'Phasic(-)' 
}

print("Initializing Global Reference Brain Canvas for Raw Responders...")
if not os.path.exists(TEMPLATE_BRAIN_PATH):
    raise FileNotFoundError(f"Missing mandatory anatomical canvas template: {TEMPLATE_BRAIN_PATH}")

nii_obj = nib.load(TEMPLATE_BRAIN_PATH)
template_img = nii_obj.get_fdata()

bg_canvas = np.max(template_img, axis=2).T 
plt.ioff()

print("Generating Pre-Clustering Raw Responder Map Overlays...")

for expt_ID in EXPT_FISH_LIST:
    print(f"\nEvaluating raw responder maps for: {expt_ID}")
    fish_dir = BASE_DIR / PROJ_ID / expt_ID
    
    vox_path = fish_dir / "medoids_template_vox.npy"
    if not vox_path.exists():
        print(f"  Warning: {vox_path} missing. Skipping this fish.")
        continue
    medoids_vox = np.load(vox_path)
    

    valid_registration_mask = (medoids_vox >= 0).all(axis=1)
    
    fig, axes = plt.subplots(2, 2, figsize=(14, 14))
    plt.suptitle(f"Pre-Clustering Whole Responder Distributions\n{expt_ID} (Tonic: p95 | Phasic: ±0.25 Cutoff)", fontsize=15, y=0.98)
    axes = axes.flatten()
    
    for idx, category in enumerate(RESPONSE_TYPES):
        ax = axes[idx]
        
        is_tonic = "tonic" in category
        if is_tonic:
            idx_path = fish_dir / f"{category}_glm_iaaft_nullp95_idxs.npy"
            if not idx_path.exists():
                ax.text(0.5, 0.5, f"Missing index file:\n{category}_nullp95", ha='center', va='center', color='gray')
                ax.axis("off")
                continue
            raw_responder_idxs = np.load(idx_path)
        else:
            dprime_path = fish_dir / "phasic_dprime_cells_raw.npy"
            if not dprime_path.exists():
                ax.text(0.5, 0.5, f"Missing dprime file:\nphasic_dprime_cells_raw.npy", ha='center', va='center', color='gray')
                ax.axis("off")
                continue
            dprime = np.load(dprime_path)
            if "pos" in category:
                raw_responder_idxs = np.where(dprime >= 0.25)[0]
            else:
                raw_responder_idxs = np.where(dprime <= -0.25)[0]
        
        valid_raw_idxs = raw_responder_idxs[valid_registration_mask[raw_responder_idxs]]
        total_cells_found = len(valid_raw_idxs)
        
        ax.imshow(bg_canvas, cmap="gray", origin="upper")
        
        if total_cells_found == 0:
            ax.text(0.5, 0.5, f"No Responding Cells\n(n = 0)", ha='center', va='center', color='gray')
            ax.set_title(LABEL_TITLES[category], fontsize=12)
            ax.axis("off")
            continue
            
        coords = medoids_vox[valid_raw_idxs]
        
        ax.scatter(
            coords[:, 0], 
            coords[:, 1], 
            color=CATEGORY_COLORS[category],
            s=2.5,
            alpha=0.6, 
            edgecolors='none'
        )

        
        ax.set_title(f"{LABEL_TITLES[category]} Responder Pool (n = {total_cells_found} cells)", fontsize=13)
        ax.set_xlim(0, bg_canvas.shape[1])
        ax.set_ylim(bg_canvas.shape[0], 0)
        ax.axis("off")

    fig.subplots_adjust(wspace=0.3, hspace=0.3)
    fish_summary_dir = os.path.join(SUMMARY_PLOTS_DIR, expt_ID)
    os.makedirs(fish_summary_dir, exist_ok=True)
    
    output_path = os.path.join(fish_summary_dir, "whole_pool_responder_distribution_1.png")
    fig.savefig(output_path, dpi=200, bbox_inches='tight', pad_inches=0.4)
    print(f"Successfully saved unified raw pool grid: {output_path}")
    
    plt.close(fig)
    gc.collect()

plt.ion()

In [ ]:
#Double check that the pipeline created the folders
import os
from pathlib import Path

check_fish_id = "251008_hcrt-trpv1_huc-h2b-g8m_csn_10uM_fish4"
target_fish_dir = Path(BASE_DIR) / PROJ_ID / check_fish_id

print(f"{check_fish_id}")
print(f"Does fish root folder exist? {target_fish_dir.exists()}")

if target_fish_dir.exists():
    print("\nSubdirectories inside this fish folder:")
    subdirs = [x.name for x in target_fish_dir.iterdir() if x.is_dir()]
    for s in subdirs:
        print(f" - {s}")
        
    loader_target_dir = target_fish_dir / OUTPUT_DIR
    print(f"\n Exact folder:\n   {loader_target_dir}")
    print(f"Does it exist? {loader_target_dir.exists()}")
    
    if loader_target_dir.exists():
        print("\nIndex files inside that folder:")
        files = list(loader_target_dir.glob("*.npy"))
        if files:
            for f in files[:5]:
                print(f"   - {f.name}")
        else:
            print("No .npy index files found in this folder!")


In [ ]:
#DO NOT USE
#Load output from pipeline
stim_start_post = 2700 
stim_end_post = 7200 
loaded_cluster_data = {fish: {cat: {} for cat in RESPONSE_TYPES} for fish in EXPT_FISH_LIST} 

def process_fish_analysis(fish_id): 
    fish_dir = BASE_DIR / PROJ_ID / fish_id 
    cluster_dir = fish_dir / f"FA_agglo_clustering_pct={N_COMPONENTS}_c={N_CLUSTERS}_t={N_CLUSTERS}" 
    
    if not cluster_dir.exists(): 
        print(f"Skipping {fish_id} — Clustering output directory not found.") 
        return None 
        
    fish_results = {cat: {} for cat in RESPONSE_TYPES} 
    
    for category in RESPONSE_TYPES: 
        is_tonic = "tonic" in category 
        data_filename = "f_tonic.npy" if is_tonic else "f_phasic.npy" 
        data_path = fish_dir / data_filename 
        if not data_path.exists(): 
            continue 
            
        try: 
            all_cluster_cells = [] 
            cluster_cell_mappings = {} 
            
            for c_idx in range(1, N_CLUSTERS + 1): 
                idx_file = cluster_dir / f"{category}_c{c_idx}_idxs.npy" 
                if idx_file.exists(): 
                    c_cells = np.load(idx_file) 
                    if len(c_cells) > 0: 
                        all_cluster_cells.append(c_cells) 
                        cluster_cell_mappings[c_idx] = c_cells 
                        
            if not all_cluster_cells: 
                continue 
                
            total_responders = np.concatenate(all_cluster_cells) 
            unique_responders = np.unique(total_responders) 
            
            raw_traces_mmap = np.load(data_path, mmap_mode="r") 
            pooled_post_traces = np.array(raw_traces_mmap[unique_responders, stim_start_post:stim_end_post]) 
            pooled_full_traces = np.array(raw_traces_mmap[unique_responders, 0:7200]) 
            
            cell_to_ram_idx = {cell: i for i, cell in enumerate(unique_responders)} 
            
            for c_idx in range(1, N_CLUSTERS + 1): 
                if c_idx not in cluster_cell_mappings: 
                    continue 
                    
                cluster_cells = cluster_cell_mappings[c_idx] 

                ram_indices = [cell_to_ram_idx[cell] for cell in cluster_cells] 
                
                c_post = pooled_post_traces[ram_indices] 
                c_full = pooled_full_traces[ram_indices] 
                
                z_post = zscore(c_post, axis=1) 
                z_full = zscore(c_full, axis=1) 
                
                valid_mask = np.all(np.isfinite(z_post), axis=1) & np.all(np.isfinite(z_full), axis=1) 
                
                if np.sum(valid_mask) > 0: 
                    fish_results[category][c_idx] = { 
                        'post_drug': z_post[valid_mask], 
                        'full_timeline': z_full[valid_mask] 
                    } 
        except Exception as e: 
            print(f"Error loading {category} for {fish_id}: {e}") 
            continue 
            
    return fish_id, fish_results 

print("Loading Dual-Window post-drug and full-timeline matrices from disk...") 
with ThreadPoolExecutor(max_workers=3) as executor: 
    results = executor.map(process_fish_analysis, EXPT_FISH_LIST) 
    for result in results: 
        if result is not None: 
            fish_id, fish_results = result 
            loaded_cluster_data[fish_id] = fish_results 

print("Data loading completed successfully!")


In [ ]:
bg_canvas = np.max(template_img, axis=2).T
bg_canvas = np.flipud(bg_canvas)
medoids_vox = np.load(vox_path)

In [ ]:
#USE THIS ONE
stim_start_post = 2700
stim_end_post = 7200
loaded_cluster_data = {fish: {cat: {} for cat in RESPONSE_TYPES} for fish in EXPT_FISH_LIST}

def process_fish_analysis(fish_id):
    fish_dir = BASE_DIR / PROJ_ID / fish_id

    cluster_dir = fish_dir / f'FA_agglo_clustering_f={N_COMPONENTS}_c={N_CLUSTERS}_t={N_CLUSTERS}'
    
    if not cluster_dir.exists():
        print(f"Skipping {fish_id} — Clustering output directory not found at {cluster_dir}")
        return None
    vox_path = fish_dir / "medoids_template_vox.npy"
    if not vox_path.exists():
        print(f"Skipping {fish_id} — Coordinates array missing.")
        return None
    medoids_vox = np.load(vox_path)
    
    fish_results = {cat: {} for cat in RESPONSE_TYPES}
    
    for category in RESPONSE_TYPES:
        is_tonic = "tonic" in category
        data_filename = "f_tonic.npy" if is_tonic else "f_phasic.npy"
        data_path = fish_dir / data_filename
        
        if not data_path.exists():
            continue
            
        try:
            all_cluster_cells = []
            cluster_cell_mappings = {}
            
            for c_idx in range(1, N_CLUSTERS + 1):
                idx_file = cluster_dir / f"{category}_c{c_idx}_idxs.npy"
                if idx_file.exists():
                    c_cells = np.load(idx_file)
                    if len(c_cells) > 0:
                        all_cluster_cells.append(c_cells)
                        cluster_cell_mappings[c_idx] = c_cells
                        
            if not all_cluster_cells:
                continue
                
            total_responders = np.concatenate(all_cluster_cells)
            unique_responders = np.unique(total_responders)
            
            raw_traces_mmap = np.load(data_path, mmap_mode="r")
            pooled_post_traces = np.array(raw_traces_mmap[unique_responders, stim_start_post:stim_end_post])
            pooled_full_traces = np.array(raw_traces_mmap[unique_responders, 0:7200])
            
            cell_to_ram_idx = {cell: i for i, cell in enumerate(unique_responders)}
            
            for c_idx in range(1, N_CLUSTERS + 1):
                if c_idx not in cluster_cell_mappings:
                    continue
                    
                cluster_cells = cluster_cell_mappings[c_idx]
                ram_indices = [cell_to_ram_idx[cell] for cell in cluster_cells]
                
                c_post = pooled_post_traces[ram_indices]
                c_full = pooled_full_traces[ram_indices]
                
                z_post = zscore(c_post, axis=1)
                z_full = zscore(c_full, axis=1)
                
                valid_mask = np.all(np.isfinite(z_post), axis=1) & np.all(np.isfinite(z_full), axis=1)
                
                if np.sum(valid_mask) > 0:
            
                    final_cluster_cells = cluster_cells[valid_mask]
                    
                    fish_results[category][c_idx] = {
                        'post_drug': z_post[valid_mask],
                        'full_timeline': z_full[valid_mask],
                        'cell_ids': final_cluster_cells,
                        'coords': medoids_vox[final_cluster_cells] 
                    }
                    
        except Exception as e:
            print(f"Error loading {category} for {fish_id}: {e}")
            continue
            
    return fish_id, fish_results

print("Loading Dual-Window post-drug and full-timeline matrices from disk...")
# Clear old dictionary references to prevent leaks
loaded_cluster_data = {} 

with ThreadPoolExecutor(max_workers=3) as executor:
    results = executor.map(process_fish_analysis, EXPT_FISH_LIST)
    for result in results:
        if result is not None:
            fish_id, fish_results = result
            loaded_cluster_data[fish_id] = fish_results

print("Data loading completed successfully")


In [ ]:
from concurrent.futures import ThreadPoolExecutor
import numpy as np
from scipy.stats import zscore

# Constants
STIM_START_POST = 2700
STIM_END_POST = 7200
TOTAL_TIMELINE_END = 7200

def process_fish_analysis(fish_id):
    fish_dir = BASE_DIR / PROJ_ID / fish_id
    cluster_dir = fish_dir / f'FA_agglo_clustering_f={N_COMPONENTS}_c={N_CLUSTERS}_t={N_CLUSTERS}'
    
    if not cluster_dir.exists():
        print(f"Skipping {fish_id} — Clustering output directory missing: {cluster_dir}")
        return None
    
    vox_path = fish_dir / "medoids_template_vox.npy" 
    if not vox_path.exists():
        print(f"Skipping {fish_id} — Coordinates array missing.")
        return None
    master_coordinates = np.load(vox_path) 

    fish_results = {}
    
    for category in RESPONSE_TYPES:
        fish_results[category] = {}
        is_tonic = "tonic" in category
        data_filename = "f_tonic.npy" if is_tonic else "f_phasic.npy"
        data_path = fish_dir / data_filename
        
        if not data_path.exists():
            continue

        try:
            cluster_cell_mappings = {}
            all_cluster_cells = []
            
            for c_idx in range(1, N_CLUSTERS + 1):
                idx_file = cluster_dir / f"{category}_c{c_idx}_idxs.npy"
                if idx_file.exists():
                    c_cells = np.load(idx_file)
                    if c_cells.size > 0:
                        all_cluster_cells.append(c_cells)
                        cluster_cell_mappings[c_idx] = c_cells

            if not all_cluster_cells:
                continue

            unique_responders = np.unique(np.concatenate(all_cluster_cells))
            
            raw_traces_mmap = np.load(data_path, mmap_mode="r")
            pooled_post_traces = np.array(raw_traces_mmap[unique_responders, STIM_START_POST:STIM_END_POST])
            pooled_full_traces = np.array(raw_traces_mmap[unique_responders, 0:TOTAL_TIMELINE_END])
   
            cell_to_ram_idx = {cell: i for i, cell in enumerate(unique_responders)}
            
            for c_idx, cluster_cells in cluster_cell_mappings.items():
                ram_indices = [cell_to_ram_idx[cell] for cell in cluster_cells]
                
                c_post = pooled_post_traces[ram_indices]
                c_full = pooled_full_traces[ram_indices]
                
                z_post = zscore(c_post, axis=1)
                z_full = zscore(c_full, axis=1)
    
                valid_mask = np.all(np.isfinite(z_post), axis=1) & np.all(np.isfinite(z_full), axis=1)
                
                if np.any(valid_mask):
                    final_cluster_cells = cluster_cells[valid_mask]
                    fish_results[category][c_idx] = {
                        'post_drug': z_post[valid_mask],
                        'full_timeline': z_full[valid_mask],
                        'cell_ids': final_cluster_cells,
                        'coords': master_coordinates[final_cluster_cells]
                    }
        except Exception as e:
            print(f"Error loading {category} for {fish_id}: {e}")
            continue
            
    return fish_id, fish_results

print("Loading Dual-Window post-drug and full-timeline matrices from disk...")
loaded_cluster_data = {}

with ThreadPoolExecutor(max_workers=3) as executor:
    results = executor.map(process_fish_analysis, EXPT_FISH_LIST)
    for result in results:
        if result is not None:
            fish_id, fish_results = result
            loaded_cluster_data[fish_id] = fish_results

print("Data loading completed successfully.")


In [ ]:
import ants
# Load your unified template once for the backdrop
template_path = "/mnt/storage-raid10/Yun/analysis_output/registration/template_mean_brain.nii.gz"
template_img = ants.image_read(template_path).numpy()
bg_canvas = np.flipud(np.max(template_img, axis=2).T)

# Pick an example to plot
target_fish = EXPT_FISH_LIST[0]
target_category = "phasic_pos"
target_cluster = 1

if target_cluster in loaded_cluster_data[target_fish][target_category]:
    cluster_meta = loaded_cluster_data[target_fish][target_category][target_cluster]
    
    # Read the pre-filtered variables directly from your dictionary data model
    coords = cluster_meta['coords'] 
    
    fig, ax = plt.subplots(figsize=(6, 10))
    ax.imshow(bg_canvas, cmap="gray", origin="upper")
    
    # Standard upright projection math mapping
    ax.scatter(
        coords[:, 0], 
        bg_canvas.shape[0] - coords[:, 1], 
        color="red", 
        s=2.0, 
        alpha=0.5, 
        edgecolors='none'
    )
    
    ax.set_title(f"{target_fish}\n{target_category} - Cluster {target_cluster} (n={len(coords)})")
    ax.axis("off")
    plt.show()


In [ ]:
# USE THIS ONE
import os
import gc
import numpy as np
import nibabel as nib
import matplotlib.pyplot as plt

TEMPLATE_BRAIN_PATH = "/mnt/storage-raid10/Yun/analysis_output/registration/template_mean_brain.nii.gz" 
SUMMARY_PLOTS_DIR = "/ssd-pool/james/lightsheet/Zebrafish-whole-brain-analysis/unsupervised_plots_summary" 

CATEGORY_COLORS = { 
    'tonic_pos': 'orange', 
    'tonic_neg': 'blue', 
    'phasic_pos': 'red', 
    'phasic_neg': 'purple' 
} 

LABEL_TITLES = { 
    'tonic_pos': 'Tonic(+)', 
    'tonic_neg': 'Tonic(-)', 
    'phasic_pos': 'Phasic(+)', 
    'phasic_neg': 'Phasic(-)' 
} 

print("Initializing Global Reference Brain Canvas for Raw Responders...") 
if not os.path.exists(TEMPLATE_BRAIN_PATH): 
    raise FileNotFoundError(f"Missing mandatory anatomical canvas template: {TEMPLATE_BRAIN_PATH}") 

nii_obj = nib.load(TEMPLATE_BRAIN_PATH) 
template_img = nii_obj.get_fdata() 

bg_canvas = np.max(template_img, axis=2).T 

plt.ioff() 
print("Generating Pre-Clustering Raw Responder Map Overlays...") 

for expt_ID in EXPT_FISH_LIST: 
    print(f"\nEvaluating raw responder maps for: {expt_ID}") 
    fish_dir = BASE_DIR / PROJ_ID / expt_ID 
    vox_path = fish_dir / "medoids_template_vox.npy" 
    
    if not vox_path.exists(): 
        print(f" Warning: {vox_path} missing. Skipping this fish.") 
        continue 
        
    medoids_vox = np.load(vox_path) 
    valid_registration_mask = (medoids_vox >= 0).all(axis=1) 
    
    fig, axes = plt.subplots(2, 2, figsize=(14, 14)) 
    plt.suptitle(f"Pre-Clustering Whole Responder Distributions\n{expt_ID} (Top {100-PERCENTILE_CUTOFF}% Cutoff)", fontsize=15, y=0.98) 
    axes = axes.flatten() 
    
    for idx, category in enumerate(RESPONSE_TYPES): 
        ax = axes[idx] 
        

        cluster_dir = fish_dir / f'FA_agglo_clustering_f={N_COMPONENTS}_c={N_CLUSTERS}_t={N_CLUSTERS}'
        
        pipeline_idxs = [] 
        for c_idx in range(1, N_CLUSTERS + 1): 
            idx_file = cluster_dir / f"{category}_c{c_idx}_idxs.npy" 
            if idx_file.exists(): 
                pipeline_idxs.append(np.load(idx_file)) 
                
        if len(pipeline_idxs) == 0: 
            ax.text(0.5, 0.5, f"Missing cluster files for:\n{category}", ha='center', va='center', color='gray') 
            ax.axis("off") 
            continue 
            
        raw_responder_idxs = np.concatenate(pipeline_idxs) 
        valid_raw_idxs = raw_responder_idxs[valid_registration_mask[raw_responder_idxs]] 
        total_cells_found = len(valid_raw_idxs) 
        
        ax.imshow(bg_canvas, cmap="gray", origin="upper") 
        
        if total_cells_found == 0: 
            ax.text(0.5, 0.5, f"No Responding Cells\n(n = 0)", ha='center', va='center', color='gray') 
            ax.set_title(LABEL_TITLES[category], fontsize=12) 
            ax.axis("off") 
            continue 
            
        coords = medoids_vox[valid_raw_idxs] 
    
        ax.scatter( 
            coords[:, 0], 
            coords[:, 1], 
            color=CATEGORY_COLORS[category], 
            s=1.2, 
            alpha=0.35, 
            edgecolors='none' 
        ) 
        
        ax.set_title(f"{LABEL_TITLES[category]} Responder Pool (n = {total_cells_found} cells)", fontsize=13) 
        ax.set_xlim(0, bg_canvas.shape[1]) 
     
        ax.set_ylim(bg_canvas.shape[0], 0) 
        ax.axis("off") 
        
    fig.subplots_adjust(wspace=0.3, hspace=0.3) 
    fish_summary_dir = os.path.join(SUMMARY_PLOTS_DIR, expt_ID) 
    os.makedirs(fish_summary_dir, exist_ok=True) 
    
    output_path = os.path.join(fish_summary_dir, f"whole_distribution_plot_pct_{PERCENTILE_CUTOFF}.png") 
    fig.savefig(output_path, dpi=200, bbox_inches='tight', pad_inches=0.4) 
    print(f" Successfully saved unified raw pool grid -> {output_path}") 
    plt.close(fig) 
    gc.collect() 

plt.ion()


# Step 2 - Heatmap

In [ ]:
#Create Heatmap
LABEL_TITLES = { 
    'tonic_pos': 'Tonic(+)', 
    'tonic_neg': 'Tonic(-)', 
    'phasic_pos': 'Phasic(+)', 
    'phasic_neg': 'Phasic(-)' 
}
CMAP = 'hot'
CLUSTER_COLORS = plt.cm.get_cmap('tab20', N_CLUSTERS)

def plot_4label_raster_fast(expt_ID):
    if expt_ID not in loaded_cluster_data:
        print(f"Skipping {expt_ID} — No data found in memory.")
        return None
        
    fig, axs = plt.subplots(2, 2, figsize=(16, 10))
    plt.suptitle(f'Clustered Neural Activity Heatmaps (Full Window)\n{expt_ID}', fontsize=16)
    
    fish_data = loaded_cluster_data[expt_ID]
    
    for idx, label in enumerate(RESPONSE_TYPES):
        ax = axs[idx // 2, idx % 2]
        cat_clusters = fish_data.get(label, {})
        
        trace_list = []
        cluster_id_list = []
    
        for c_idx in range(1, N_CLUSTERS + 1):
            if c_idx in cat_clusters:
                cluster_package = cat_clusters[c_idx]
                z_traces = cluster_package['full_timeline']
                
                trace_list.append(z_traces)
                cluster_id_list.extend([c_idx] * len(z_traces))
        
        if not trace_list:
            ax.text(0.5, 0.5, f"No responding cells found\nfor {LABEL_TITLES[label]}", 
                    ha='center', va='center', fontsize=12, color='gray')
            ax.set_title(LABEL_TITLES[label], fontsize=14)
            continue
            
        all_traces = np.vstack(trace_list)
        cluster_ids = np.array(cluster_id_list)
        
        sort_order = np.argsort(cluster_ids)
        sorted_traces = all_traces[sort_order][:, 0:]
        sorted_cluster_ids = cluster_ids[sort_order]
        
        im = ax.imshow(sorted_traces, aspect='auto', cmap=CMAP, interpolation='none', vmin=-3, vmax=6)
        
        ax.set_xlim(0, 7200)
        ax.set_title(f"{LABEL_TITLES[label]} (n = {sorted_traces.shape[0]} cells)", fontsize=14)
        ax.set_ylabel("Cells (Grouped by Cluster)", fontsize=12)
        ax.set_xlabel("All Volumes (Frames 0 - 7200)", fontsize=12)
        
        inset_ax = inset_axes(ax, width="2%", height="100%", loc='center left', 
                              bbox_to_anchor=(0.015, 0, 1, 1), bbox_transform=ax.transAxes, borderpad=0)
        cluster_colors = [CLUSTER_COLORS(c - 1) for c in sorted_cluster_ids]
        color_strip = np.array(cluster_colors).reshape(len(cluster_colors), 1, 4)
        inset_ax.imshow(color_strip, aspect='auto')
        inset_ax.axis('off')
        
        cbar_ax = inset_axes(ax, width="1.5%", height="70%", loc='right', 
                             bbox_to_anchor=(0, 0.15, 1, 1), bbox_transform=ax.transAxes, borderpad=0)
        plt.colorbar(im, cax=cbar_ax)
        
  
        for stim_time in [2700,5400]:
            ax.axvline(x=stim_time, color='white', linestyle='--', linewidth=1.0, alpha=0.8)
            
    fig.tight_layout(rect=[0, 0.03, 1, 0.95])
    fig.subplots_adjust(wspace=0.2, hspace=0.25)
    return fig


In [ ]:
#save heatmaps to a summary directory for easy access and reduces file size 
# where they are saved:
SUMMARY_PLOTS_DIR = os.path.join("/ssd-pool/james/lightsheet/Zebrafish-whole-brain-analysis/unsupervised_plots_summary")
os.makedirs(SUMMARY_PLOTS_DIR, exist_ok=True)

plt.ioff()
for expt_ID in EXPT_FISH_LIST:
    print(f"Generating heatmap for {expt_ID}...")  # Fixed space
    
    fig = plot_4label_raster_fast(expt_ID)

    if fig is not None:
        output_path = os.path.join(SUMMARY_PLOTS_DIR, f"{expt_ID}_pre_drug_heatmap_raster.png")
        
        if fig is not None:
            fig.savefig(output_path, dpi=200, bbox_inches='tight')
            
    if fig is not None:
        plt.close(fig)

plt.ion()



# Step 3 - Visualizing the spatial location of cells

In [ ]:
#this one
TEMPLATE_BRAIN_PATH = "/mnt/storage-raid10/Yun/analysis_output/registration/template_mean_brain.nii.gz"
SUMMARY_PLOTS_DIR = "/ssd-pool/james/lightsheet/Zebrafish-whole-brain-analysis/unsupervised_plots_summary"
COLOR_PALETTE = plt.cm.get_cmap('tab10', N_CLUSTERS)
if not os.path.exists(TEMPLATE_BRAIN_PATH):
    raise FileNotFoundError(f"Missing mandatory anatomical canvas template: {TEMPLATE_BRAIN_PATH}")

nii_obj = nib.load(TEMPLATE_BRAIN_PATH)
template_img = nii_obj.get_fdata()

bg_canvas = np.max(template_img, axis=2).T
bg_canvas = np.flipud(bg_canvas)
plt.ioff()

for expt_ID in EXPT_FISH_LIST:
    if expt_ID not in loaded_cluster_data:
        continue
        
    print(f"\nProjecting anatomical spatial profiles for: {expt_ID}")
    fish_dir = BASE_DIR / PROJ_ID / expt_ID
    
    vox_path = fish_dir / "medoids_template_vox.npy"
    if not vox_path.exists():
        print(f"  Missing template voxels for {expt_ID}. Skipping.")
        continue
        
    medoids_vox = np.load(vox_path)
    valid_registration_mask = (medoids_vox >= 0).all(axis=1)

    cluster_dir = fish_dir / f'FA_agglo_clustering_f={N_COMPONENTS}_c={N_CLUSTERS}_t={N_CLUSTERS}'
    
    for category in RESPONSE_TYPES:
        target_subfolder = os.path.join(SUMMARY_PLOTS_DIR, expt_ID, category)
        os.makedirs(target_subfolder, exist_ok=True)
        

        valid_cluster_files = []
        for c_idx in range(1, N_CLUSTERS + 1):
            idx_file = cluster_dir / f"{category}_c{c_idx}_idxs.npy"
            if idx_file.exists():
                cluster_indices = np.load(idx_file)
                if len(cluster_indices) > 0:
                    valid_cluster_indices = cluster_indices[valid_registration_mask[cluster_indices]]
                    if len(valid_cluster_indices) > 0:
                        valid_cluster_files.append((c_idx, idx_file, valid_cluster_indices))
                        
        n_valid_panels = len(valid_cluster_files)
        if n_valid_panels == 0:
            continue
     
        n_cols = min(6, n_valid_panels)
        n_rows = int(np.ceil(n_valid_panels / n_cols))
        
        fig, axes = plt.subplots(n_rows, n_cols, figsize=(6 * n_cols, 6 * n_rows), squeeze=False)
        plt.suptitle(f"Spatial Distribution Plot", fontsize=15, y=0.98)
        axes = axes.flatten()
        
        plotted_panels = 0
        

        for c_idx, idx_file, valid_cluster_indices in valid_cluster_files:
  
            if plotted_panels >= len(axes):
                print(f"Omitted extra cluster panel {c_idx} to avoid array out-of-bounds")
                break
                
 
            coords = medoids_vox[valid_cluster_indices]
            
            ax = axes[plotted_panels]
  
            ax.imshow(bg_canvas, cmap="gray", origin="lower")
 
            ax.scatter(
                coords[:, 0], 
                coords[:, 1], 
                color=COLOR_PALETTE(c_idx - 1),
                s=2.5, 
                alpha=0.6, 
                edgecolors='none'
            )
            
            ax.set_title(f"Cluster {c_idx} (n={len(valid_cluster_indices)} cells)", fontsize=11)
            ax.set_xlim(0, bg_canvas.shape[1])
            ax.set_ylim(0, bg_canvas.shape[0]) 
            ax.axis("off")
            ax.text(0.05, 0.08, f"C{c_idx}", 
                    transform=ax.transAxes, 
                    color="white", 
                    fontsize=20, 
                    weight="bold", 
                    ha="left", 
                    va="bottom")
            plotted_panels += 1
            

        for empty_idx in range(plotted_panels, len(axes)):
            fig.delaxes(axes[empty_idx])
            
        if plotted_panels > 0:
            fig.subplots_adjust(wspace=0.1, hspace=0.1)
            output_path = os.path.join(target_subfolder, "spatial_distrbution_plot.png")
            fig.savefig(output_path, dpi=200, bbox_inches='tight', pad_inches=0.5)
            print(f"Saved plot for: {output_path}")
            
        plt.close(fig)
        
    gc.collect()

plt.ion()
print("\nAll plots rendered")

In [ ]:
# Pipeline + Data Load for quick test keep

import sys
import pandas as pd
import joblib
from tqdm import tqdm
from pathlib import Path
from sklearn.decomposition import TruncatedSVD

import os
import gc
from pathlib import Path
import numpy as np
import joblib
from scipy.stats import zscore
import matplotlib.pyplot as plt
import nibabel as nib
from sklearn.decomposition import FactorAnalysis
from sklearn.cluster import AgglomerativeClustering
from concurrent.futures import ProcessPoolExecutor 
import ants
from sklearn.decomposition import FactorAnalysis

from concurrent.futures import ThreadPoolExecutor
from mpl_toolkits.axes_grid1.inset_locator import inset_axes

BASE_DIR = Path("/mnt/storage-raid10/Yun/analysis_output/chemogenetic")
PROJ_ID = "hcrt-trpv1_huc-h2b-g8m_csn_120min"
PROJ_CTRL = "huc-h2b-g8m_csn_120min"
TEMPLATE_BRAIN_PATH = "/mnt/storage-raid10/Yun/analysis_output/registration/template_mean_brain.nii.gz"
SUMMARY_PLOTS_DIR = "/ssd-pool/james/lightsheet/Zebrafish-whole-brain-analysis/unsupervised_plots_summary"
stim_start_post = 2700
stim_end_post = 7200
PERCENTILE_CUTOFF = 90
'''
EXPT_FISH_LIST = [
    "251008_hcrt-trpv1_huc-h2b-g8m_csn_10uM_fish4",
    "251102_hcrt-trpv1_huc-h2b-g8m_csn_10uM_fish1",
    "251102_hcrt-trpv1_huc-h2b-g8m_csn_10uM_fish2",
    "251210_hcrt-trpv1_huc-h2b-g8m_csn_10uM_fish1",
    "251210_hcrt-trpv1_huc-h2b-g8m_csn_10uM_fish2", 
    "251210_hcrt-trpv1_huc-h2b-g8m_csn_10uM_fish3",
    "260514_hcrt-trpv1_huc-h2b-g8m_csn_10uM_fish1",
    "260514_hcrt-trpv1_huc-h2b-g8m_csn_10uM_fish2",
    "260515_hcrt-trpv1_huc-h2b-g8m_csn_10uM_fish1"
]

'''

RESPONSE_TYPES = [
   "tonic_pos",
   "tonic_neg",

                    "phasic_pos",
                    "phasic_neg"
                 ]

DIR_ANTS_OUTPUT = str(BASE_DIR)

example_fish = '251008_hcrt-trpv1_huc-h2b-g8m_csn_10uM_fish4' #used for testing

N_COMPONENTS = 15
N_CLUSTERS = 7

EXPT_FISH_LIST = [
    "251008_hcrt-trpv1_huc-h2b-g8m_csn_10uM_fish4"
]


CATEGORY_COLORS = {'tonic_pos': 'orange', 'tonic_neg': 'blue', 'phasic_pos': 'red', 'phasic_neg': 'purple'} 
LABEL_TITLES = {'tonic_pos': 'Tonic(+)', 'tonic_neg': 'Tonic(-)', 'phasic_pos': 'Phasic(+)', 'phasic_neg': 'Phasic(-)'} 

from multiprocessing import Pool, cpu_count

def process_single_experiment(args):
    expt_ID, PERCENTILE_CUTOFF, PROJ_ID, DIR_ANTS_OUTPUT, medoid_types, stim_start, stim_end, use_qc, t, save_dendrogram, max_cells, n_components = args
    
    dir_expt = os.path.join(DIR_ANTS_OUTPUT, PROJ_ID, expt_ID)
    OUTPUT_DIR = os.path.join(dir_expt, f"FA_agglo_clustering_pct={PERCENTILE_CUTOFF}_c={t}")
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    
    loaded_traces = {}

    for label in medoid_types:
        is_tonic = "tonic" in label
        data_filename = "f_tonic.npy" if is_tonic else "f_phasic.npy"
        data_path = os.path.join(dir_expt, data_filename)
        
        if not os.path.exists(data_path):
            continue

        if data_filename not in loaded_traces:
            loaded_traces[data_filename] = np.load(data_path)
        traces = loaded_traces[data_filename]

        
        if is_tonic:
            tonic_score = np.var(traces[:, stim_start:stim_end], axis=1)
            
            if 'pos' in label:
                tonic_cutoff = np.percentile(tonic_score, PERCENTILE_CUTOFF)
                idxs = np.where(tonic_score >= tonic_cutoff)[0]
            else:
                tonic_cutoff = np.percentile(tonic_score, 100 - PERCENTILE_CUTOFF)
                idxs = np.where(tonic_score <= tonic_cutoff)[0]
            
        else:
            dprime_path = os.path.join(dir_expt, "phasic_dprime_cells_raw.npy")
            if not os.path.exists(dprime_path):
                continue
            dprime = np.load(dprime_path)
            
            
            if "pos" in label:
                top_percentile_cutoff = np.nanpercentile(dprime, PERCENTILE_CUTOFF)
          
                idxs = np.where(dprime >= top_percentile_cutoff)[0]
            else:
                top_percentile_cutoff = np.nanpercentile(dprime, 100 - PERCENTILE_CUTOFF)
        
                idxs = np.where(dprime <= top_percentile_cutoff)[0]

          
            print(f"\n[{label} COMPUTE CHECK]")
            print(f"Top percentile value calculated: {top_percentile_cutoff}")
            print(f"Extracted array length before check: {len(idxs)}")
            
            idxs = idxs[idxs < traces.shape[0]]

            print(f" [{label}] Cells after bounds check: {len(idxs)}")

        if len(idxs) == 0:
            print(f" Warning: [{expt_ID} - {label}] No cells passed the percentile threshold.")
            continue
            
        trace_subset = traces[idxs, stim_start:stim_end]

        row_variances = np.var(trace_subset, axis=1)
        nonzero_variance_mask = row_variances > 1e-8
        
        if np.sum(nonzero_variance_mask) < 4:
            print(f" Warning: [{expt_ID} - {label}] Dropped. Only {np.sum(nonzero_variance_mask)} cells had non-zero variance during the stim window.")
            continue
            
      
        trace_subset = trace_subset[nonzero_variance_mask]
        idxs = idxs[nonzero_variance_mask]
        
        z_traces = zscore(trace_subset, axis=1)
        valid_mask = np.all(np.isfinite(z_traces), axis=1)
        
        if np.sum(valid_mask) < 4:
            print(f" Warning: [{expt_ID} - {label}] Dropped. Z-scoring still resulted in non-finite values.")
            continue
            
        z_traces = z_traces[valid_mask]
        idxs = idxs[valid_mask]

        MAX_TARGET_CELLS = 50000
        if len(idxs) > MAX_TARGET_CELLS:
    
            metric_arr = tonic_score[idxs] if is_tonic else dprime[idxs]
            
            if "pos" in label:
                top_orders = np.argsort(metric_arr)[-MAX_TARGET_CELLS:]
            else:
                top_orders = np.argsort(metric_arr)[:MAX_TARGET_CELLS]
        
            z_traces = z_traces[top_orders]
            idxs = idxs[top_orders]



        print(f"[{expt_ID} - {label}] Processing FA + Clustering with {z_traces.shape[0]} cells...")

        try:
            fa = FactorAnalysis(n_components=n_components, random_state=0)
            latent = fa.fit_transform(z_traces)
            fa_model_path = os.path.join(OUTPUT_DIR, f"FA_model_{label}.joblib")
            joblib.dump(fa, fa_model_path)

            clusterer = AgglomerativeClustering(n_clusters=t, linkage='ward')
            cluster_labels = clusterer.fit_predict(latent)

            for c_idx in range(1, t + 1):

                c_mask = (cluster_labels == (c_idx - 1))
                cluster_cell_ids = idxs[c_mask]
                out_idx_path = os.path.join(OUTPUT_DIR, f"{label}_c{c_idx}_idxs.npy")
                np.save(out_idx_path, cluster_cell_ids)

            del z_traces, latent, trace_subset, idxs
        except Exception as e:
            print(f"Failure processing cluster execution line for {label}: {e}")
            continue
        
    del loaded_traces
    gc.collect()
    return expt_ID

def hierarchical_clustering_parallel(
    EXPT_FISH_LIST, PERCENTILE_CUTOFF, PROJ_ID, DIR_ANTS_OUTPUT, RESPONSE_TYPES, 
    stim_start, stim_end, use_qc, t, save_dendrogram, max_cells, n_components
):
    tasks = [(
        expt_ID, PERCENTILE_CUTOFF, PROJ_ID, DIR_ANTS_OUTPUT, RESPONSE_TYPES, 
        stim_start, stim_end, use_qc, t, save_dendrogram, max_cells, n_components
    ) for expt_ID in EXPT_FISH_LIST]
    
    num_workers = max(1, int(cpu_count() * 0.75)) 
    print(f"Starting parallel execution with {num_workers} workers.")
    
    with Pool(processes=num_workers) as pool:
        results = list(tqdm(pool.imap_unordered(process_single_experiment, tasks), total=len(tasks), desc="Parallel clustering"))        
    return results

if __name__ == "__main__":
    print("STEP 1: Running Parallel Agglomerative Clustering")

    hierarchical_clustering_parallel(
        EXPT_FISH_LIST=EXPT_FISH_LIST, PERCENTILE_CUTOFF=PERCENTILE_CUTOFF, PROJ_ID=PROJ_ID,
        DIR_ANTS_OUTPUT=DIR_ANTS_OUTPUT, RESPONSE_TYPES=RESPONSE_TYPES,
        stim_start=2700, stim_end=7200, use_qc = False,  t=N_CLUSTERS, n_components=N_COMPONENTS, 
        save_dendrogram=False,
        max_cells=50000
    )

loaded_cluster_data = {}
def process_fish_analysis(fish_id):
    fish_dir = BASE_DIR / PROJ_ID / fish_id

    cluster_dir = fish_dir / f'FA_agglo_clustering_pct={PERCENTILE_CUTOFF}_c={N_CLUSTERS}'
        
    if not cluster_dir.exists():
        print(f"Skipping {fish_id} — Clustering output directory not found at {cluster_dir}")
        return None
    vox_path = fish_dir / "medoids_template_vox.npy"
    if not vox_path.exists():
        print(f"Skipping {fish_id} — Coordinates array missing.")
        return None
    medoids_vox = np.load(vox_path)
    
    fish_results = {cat: {} for cat in RESPONSE_TYPES}
    
    for category in RESPONSE_TYPES:
        is_tonic = "tonic" in category
        data_filename = "f_tonic.npy" if is_tonic else "f_phasic.npy"
        data_path = fish_dir / data_filename
        
        if not data_path.exists():
            print(f'{data_path} does not exist')
            continue
            
        try:
            all_cluster_cells = []
            cluster_cell_mappings = {}
            
            for c_idx in range(1, N_CLUSTERS + 1):
                idx_file = cluster_dir / f"{category}_c{c_idx}_idxs.npy"
                if idx_file.exists():
                    c_cells = np.load(idx_file)
                    if len(c_cells) > 0:
                        all_cluster_cells.append(c_cells)
                        cluster_cell_mappings[c_idx] = c_cells
                        
            if not all_cluster_cells:
                continue
                
            unique_responders = np.unique(np.concatenate(all_cluster_cells))
                
            raw_traces_mmap = np.load(data_path, mmap_mode="r")
            pooled_post_traces = np.array(raw_traces_mmap[unique_responders, stim_start_post:stim_end_post])
            pooled_full_traces = np.array(raw_traces_mmap[unique_responders, 0:7200])
            cell_to_ram_idx = {cell: i for i, cell in enumerate(unique_responders)}
            
            for c_idx in range(1, N_CLUSTERS + 1):
                if c_idx not in cluster_cell_mappings:
                    continue
                    
                cluster_cells = cluster_cell_mappings[c_idx]
                ram_indices = [cell_to_ram_idx[cell] for cell in cluster_cells]
                
                c_post = pooled_post_traces[ram_indices]
                c_full = pooled_full_traces[ram_indices]
                
                z_post = zscore(c_post, axis=1)
                z_full = zscore(c_full, axis=1)
                
                valid_mask = np.all(np.isfinite(z_post), axis=1) & np.all(np.isfinite(z_full), axis=1)
                
                if np.sum(valid_mask) > 0:
            
                    final_cluster_cells = cluster_cells[valid_mask]
                    
                    fish_results[category][c_idx] = {
                        'post_drug': z_post[valid_mask],
                        'full_timeline': z_full[valid_mask],
                        'cell_ids': final_cluster_cells,
                        'coords': medoids_vox[final_cluster_cells] 
                    }
                    
        except Exception as e:
            print(f"Error loading {category} for {fish_id}: {e}")
            import traceback 
            print(f"\nCRITICAL CRASH inside {category} for fish {expt_ID}:")
            print(f"Error Type: {str(e)}")
            traceback.print_exc()
            continue
            
    return fish_id, fish_results


with ThreadPoolExecutor(max_workers=3) as executor:
    results = executor.map(process_fish_analysis, EXPT_FISH_LIST)
    for result in results:
        if result is not None:
            fish_id, fish_results = result
            loaded_cluster_data[fish_id] = fish_results

print("Data loading completed successfully")


In [ ]:
#Plot Whole DIstribution PLots
# USE THIS
if not os.path.exists(TEMPLATE_BRAIN_PATH): 
    raise FileNotFoundError(f"Missing mandatory anatomical canvas template: {TEMPLATE_BRAIN_PATH}") 

nii_obj = nib.load(TEMPLATE_BRAIN_PATH) 
template_img = nii_obj.get_fdata() 

bg_canvas = np.max(template_img, axis=2).T 

plt.ioff() 

for expt_ID in EXPT_FISH_LIST: 
    print(f"\nEvaluating raw responder maps for: {expt_ID}") 
    fish_dir = BASE_DIR / PROJ_ID / expt_ID 
    vox_path = fish_dir / "medoids_template_vox.npy" 
    
    if not vox_path.exists(): 
        print(f" Warning: {vox_path} missing. Skipping this fish.") 
        continue 
        
    medoids_vox = np.load(vox_path) 
    valid_registration_mask = (medoids_vox >= 0).all(axis=1) 
    
    fig, axes = plt.subplots(2, 2, figsize=(14, 14)) 
    plt.suptitle(f"Pre-Clustering Whole Responder Distributions\n{expt_ID} (Top {100-PERCENTILE_CUTOFF}% Cutoff)", fontsize=15, y=0.98)
    axes = axes.flatten() 
    
    for idx, category in enumerate(RESPONSE_TYPES): 
        ax = axes[idx] 
        

        cluster_dir = fish_dir / f'FA_agglo_clustering_pct={PERCENTILE_CUTOFF}_c={N_CLUSTERS}'
        pipeline_idxs = [] 
        for c_idx in range(1, N_CLUSTERS + 1): 
            idx_file = cluster_dir / f"{category}_c{c_idx}_idxs.npy" 
            if idx_file.exists(): 
                pipeline_idxs.append(np.load(idx_file)) 
                
        if len(pipeline_idxs) == 0: 
            ax.text(0.5, 0.5, f"Missing cluster files for:\n{category}", ha='center', va='center', color='gray') 
            ax.axis("off") 
            continue 
            
        raw_responder_idxs = np.concatenate(pipeline_idxs) 
        valid_raw_idxs = raw_responder_idxs[valid_registration_mask[raw_responder_idxs]] 
        total_cells_found = len(valid_raw_idxs) 
        
        ax.imshow(bg_canvas, cmap="gray", origin="upper") 
        
        if total_cells_found == 0: 
            ax.text(0.5, 0.5, f"No Responding Cells\n(n = 0)", ha='center', va='center', color='gray') 
            ax.set_title(LABEL_TITLES[category], fontsize=12) 
            ax.axis("off") 
            continue 
            
        coords = medoids_vox[valid_raw_idxs] 
    
        ax.scatter( 
            coords[:, 0], 
            coords[:, 1], 
            color=CATEGORY_COLORS[category], 
            s=1.2, 
            alpha=0.35, 
            edgecolors='none' 
        ) 
        
        ax.set_title(f"{LABEL_TITLES[category]} Responder Pool (n = {total_cells_found} cells)", fontsize=13) 
        ax.set_xlim(0, bg_canvas.shape[1]) 
     
        ax.set_ylim(bg_canvas.shape[0], 0) 
        ax.axis("off") 
        
    fig.subplots_adjust(wspace=0.3, hspace=0.3) 
    fish_summary_dir = os.path.join(SUMMARY_PLOTS_DIR, expt_ID) 
    os.makedirs(fish_summary_dir, exist_ok=True) 
    
    output_path = os.path.join(fish_summary_dir, f"whole_distribution_plot_pct_{PERCENTILE_CUTOFF}.png")
    fig.savefig(output_path, dpi=200, bbox_inches='tight', pad_inches=0.4) 
    print(f" Successfully saved unified raw pool grid -> {output_path}") 
    plt.close(fig) 
    gc.collect() 

plt.ion()


In [ ]:
import sys
import pandas as pd
import joblib
from tqdm import tqdm
from pathlib import Path
from sklearn.decomposition import TruncatedSVD

import os
import gc
from pathlib import Path
import numpy as np
import joblib
from scipy.stats import zscore
import matplotlib.pyplot as plt
import nibabel as nib
from sklearn.decomposition import FactorAnalysis
from sklearn.cluster import AgglomerativeClustering
from concurrent.futures import ProcessPoolExecutor 
import ants
from sklearn.decomposition import FactorAnalysis

from concurrent.futures import ThreadPoolExecutor
from mpl_toolkits.axes_grid1.inset_locator import inset_axes

BASE_DIR = Path("/mnt/storage-raid10/Yun/analysis_output/chemogenetic")
PROJ_ID = "hcrt-trpv1_huc-h2b-g8m_csn_120min"
PROJ_CTRL = "huc-h2b-g8m_csn_120min"
TEMPLATE_BRAIN_PATH = "/mnt/storage-raid10/Yun/analysis_output/registration/template_mean_brain.nii.gz"
SUMMARY_PLOTS_DIR = "/ssd-pool/james/lightsheet/Zebrafish-whole-brain-analysis/unsupervised_plots_summary"
stim_start_post = 2700
stim_end_post = 7200
PERCENTILE_CUTOFF = 90

EXPT_FISH_LIST = [
    "251008_hcrt-trpv1_huc-h2b-g8m_csn_10uM_fish4",
    #"251102_hcrt-trpv1_huc-h2b-g8m_csn_10uM_fish1",
    #"251102_hcrt-trpv1_huc-h2b-g8m_csn_10uM_fish2",
    #"251210_hcrt-trpv1_huc-h2b-g8m_csn_10uM_fish1",
    #"251210_hcrt-trpv1_huc-h2b-g8m_csn_10uM_fish2", 
    #"251210_hcrt-trpv1_huc-h2b-g8m_csn_10uM_fish3",
    #"260514_hcrt-trpv1_huc-h2b-g8m_csn_10uM_fish1",
    #"260514_hcrt-trpv1_huc-h2b-g8m_csn_10uM_fish2",
    #"260515_hcrt-trpv1_huc-h2b-g8m_csn_10uM_fish1"
]



RESPONSE_TYPES = [
    "tonic_pos",
    "tonic_neg",
    "phasic_pos",
    "phasic_neg"
                 ]

DIR_ANTS_OUTPUT = str(BASE_DIR)


N_COMPONENTS = 15
N_CLUSTERS = 7

CATEGORY_COLORS = {'tonic_pos': 'orange', 'tonic_neg': 'blue', 'phasic_pos': 'red', 'phasic_neg': 'purple'} 
LABEL_TITLES = {'tonic_pos': 'Tonic(+)', 'tonic_neg': 'Tonic(-)', 'phasic_pos': 'Phasic(+)', 'phasic_neg': 'Phasic(-)'} 

from multiprocessing import Pool, cpu_count

def process_single_experiment(args):
    expt_ID, PERCENTILE_CUTOFF, PROJ_ID, DIR_ANTS_OUTPUT, medoid_types, stim_start, stim_end, use_qc, t, save_dendrogram, max_cells, n_components = args
    
    dir_expt = os.path.join(DIR_ANTS_OUTPUT, PROJ_ID, expt_ID)
    OUTPUT_DIR = os.path.join(dir_expt, f"FA_agglo_clustering_pct={PERCENTILE_CUTOFF}_c={t}")
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    
    loaded_traces = {}

    for label in medoid_types:
        is_tonic = "tonic" in label
        data_filename = "f_tonic.npy" if is_tonic else "f_phasic.npy"
        data_path = os.path.join(dir_expt, data_filename)
        
        if not os.path.exists(data_path):
            continue

        if data_filename not in loaded_traces:
            loaded_traces[data_filename] = np.load(data_path)
        traces = loaded_traces[data_filename]

        
        if is_tonic:
            tonic_score = np.var(traces[:, stim_start:stim_end], axis=1)
            
            if 'pos' in label:
                tonic_cutoff = np.percentile(tonic_score, PERCENTILE_CUTOFF)
                idxs = np.where(tonic_score >= tonic_cutoff)[0]
            else:
                tonic_cutoff = np.percentile(tonic_score, 100 - PERCENTILE_CUTOFF)
                idxs = np.where(tonic_score <= tonic_cutoff)[0]
            
        else:
            dprime_path = os.path.join(dir_expt, "phasic_dprime_cells_raw.npy")
            if not os.path.exists(dprime_path):
                continue
            dprime = np.load(dprime_path)
            
            
            if "pos" in label:
                top_percentile_cutoff = np.nanpercentile(dprime, PERCENTILE_CUTOFF)
          
                idxs = np.where(dprime >= top_percentile_cutoff)[0]
            else:
                top_percentile_cutoff = np.nanpercentile(dprime, 100 - PERCENTILE_CUTOFF)
        
                idxs = np.where(dprime <= top_percentile_cutoff)[0]

          
            print(f"\n[{label} COMPUTE CHECK]")
            print(f"Top percentile value calculated: {top_percentile_cutoff}")
            print(f"Extracted array length before check: {len(idxs)}")
            
            idxs = idxs[idxs < traces.shape[0]]

            print(f" [{label}] Cells after bounds check: {len(idxs)}")

        if len(idxs) == 0:
            print(f" Warning: [{expt_ID} - {label}] No cells passed the percentile threshold.")
            continue
            
        trace_subset = traces[idxs, stim_start:stim_end]

        row_variances = np.var(trace_subset, axis=1)
        nonzero_variance_mask = row_variances > 1e-8
        
        if np.sum(nonzero_variance_mask) < 4:
            print(f" Warning: [{expt_ID} - {label}] Dropped. Only {np.sum(nonzero_variance_mask)} cells had non-zero variance during the stim window.")
            continue
            
      
        trace_subset = trace_subset[nonzero_variance_mask]
        idxs = idxs[nonzero_variance_mask]
        
        z_traces = zscore(trace_subset, axis=1)
        valid_mask = np.all(np.isfinite(z_traces), axis=1)
        
        if np.sum(valid_mask) < 4:
            print(f" Warning: [{expt_ID} - {label}] Dropped. Z-scoring still resulted in non-finite values.")
            continue
            
        z_traces = z_traces[valid_mask]
        idxs = idxs[valid_mask]

        MAX_TARGET_CELLS = 50000
        if len(idxs) > MAX_TARGET_CELLS:
    
            metric_arr = tonic_score[idxs] if is_tonic else dprime[idxs]
            
            if "pos" in label:
                top_orders = np.argsort(metric_arr)[-MAX_TARGET_CELLS:]
            else:
                top_orders = np.argsort(metric_arr)[:MAX_TARGET_CELLS]
        
            z_traces = z_traces[top_orders]
            idxs = idxs[top_orders]



        print(f"[{expt_ID} - {label}] Processing FA + Clustering with {z_traces.shape[0]} cells...")

        try:
            fa = FactorAnalysis(n_components=n_components, random_state=0)
            latent = fa.fit_transform(z_traces)
            fa_model_path = os.path.join(OUTPUT_DIR, f"FA_model_{label}.joblib")
            joblib.dump(fa, fa_model_path)

            clusterer = AgglomerativeClustering(n_clusters=t, linkage='ward')
            cluster_labels = clusterer.fit_predict(latent)

            for c_idx in range(1, t + 1):

                c_mask = (cluster_labels == (c_idx - 1))
                cluster_cell_ids = idxs[c_mask]
                out_idx_path = os.path.join(OUTPUT_DIR, f"{label}_c{c_idx}_idxs.npy")
                np.save(out_idx_path, cluster_cell_ids)

            del z_traces, latent, trace_subset, idxs
        except Exception as e:
            print(f"Failure processing cluster execution line for {label}: {e}")
            continue
        
    del loaded_traces
    gc.collect()
    return expt_ID

def hierarchical_clustering_parallel(
    EXPT_FISH_LIST, PERCENTILE_CUTOFF, PROJ_ID, DIR_ANTS_OUTPUT, RESPONSE_TYPES, 
    stim_start, stim_end, use_qc, t, save_dendrogram, max_cells, n_components
):
    tasks = [(
        expt_ID, PERCENTILE_CUTOFF, PROJ_ID, DIR_ANTS_OUTPUT, RESPONSE_TYPES, 
        stim_start, stim_end, use_qc, t, save_dendrogram, max_cells, n_components
    ) for expt_ID in EXPT_FISH_LIST]
    
    num_workers = max(1, int(cpu_count() * 0.75)) 
    print(f"Starting parallel execution with {num_workers} workers.")
    
    with Pool(processes=num_workers) as pool:
        results = list(tqdm(pool.imap_unordered(process_single_experiment, tasks), total=len(tasks), desc="Parallel clustering"))        
    return results

if __name__ == "__main__":
    print("STEP 1: Running Parallel Agglomerative Clustering")

    hierarchical_clustering_parallel(
        EXPT_FISH_LIST=EXPT_FISH_LIST, PERCENTILE_CUTOFF=PERCENTILE_CUTOFF, PROJ_ID=PROJ_ID,
        DIR_ANTS_OUTPUT=DIR_ANTS_OUTPUT, RESPONSE_TYPES=RESPONSE_TYPES,
        stim_start=2700, stim_end=7200, use_qc = False,  t=N_CLUSTERS, n_components=N_COMPONENTS, 
        save_dendrogram=False,
        max_cells=50000
    )

loaded_cluster_data = {}
def process_fish_analysis(fish_id):
    fish_dir = BASE_DIR / PROJ_ID / fish_id

    cluster_dir = fish_dir / f'FA_agglo_clustering_pct={PERCENTILE_CUTOFF}_c={N_CLUSTERS}'
    cluster_dir = fish_dir / f'FA_agglo_clustering_pct={PERCENTILE_CUTOFF}_c={N_CLUSTERS}'
        
    if not cluster_dir.exists():
        print(f"Skipping {fish_id} — Clustering output directory not found at {cluster_dir}")
        return None
    vox_path = fish_dir / "medoids_template_vox.npy"
    if not vox_path.exists():
        print(f"Skipping {fish_id} — Coordinates array missing.")
        return None
    medoids_vox = np.load(vox_path)
    
    fish_results = {cat: {} for cat in RESPONSE_TYPES}
    
    for category in RESPONSE_TYPES:
        is_tonic = "tonic" in category
        data_filename = "f_tonic.npy" if is_tonic else "f_phasic.npy"
        data_path = fish_dir / data_filename
        
        if not data_path.exists():
            print(f'{data_path} does not exist')
            continue
            
        try:
            all_cluster_cells = []
            cluster_cell_mappings = {}
            
            for c_idx in range(1, N_CLUSTERS + 1):
                idx_file = cluster_dir / f"{category}_c{c_idx}_idxs.npy"
                if idx_file.exists():
                    c_cells = np.load(idx_file)
                    if len(c_cells) > 0:
                        all_cluster_cells.append(c_cells)
                        cluster_cell_mappings[c_idx] = c_cells
                        
            if not all_cluster_cells:
                continue
                
            unique_responders = np.unique(np.concatenate(all_cluster_cells))
                
            raw_traces_mmap = np.load(data_path, mmap_mode="r")
            pooled_post_traces = np.array(raw_traces_mmap[unique_responders, stim_start_post:stim_end_post])
            pooled_full_traces = np.array(raw_traces_mmap[unique_responders, 0:7200])
            cell_to_ram_idx = {cell: i for i, cell in enumerate(unique_responders)}
            
            for c_idx in range(1, N_CLUSTERS + 1):
                if c_idx not in cluster_cell_mappings:
                    continue
                    
                cluster_cells = cluster_cell_mappings[c_idx]
                ram_indices = [cell_to_ram_idx[cell] for cell in cluster_cells]
                
                c_post = pooled_post_traces[ram_indices]
                c_full = pooled_full_traces[ram_indices]
                
                z_post = zscore(c_post, axis=1)
                z_full = zscore(c_full, axis=1)
                
                valid_mask = np.all(np.isfinite(z_post), axis=1) & np.all(np.isfinite(z_full), axis=1)
                
                if np.sum(valid_mask) > 0:
            
                    final_cluster_cells = cluster_cells[valid_mask]
                    
                    fish_results[category][c_idx] = {
                        'post_drug': z_post[valid_mask],
                        'full_timeline': z_full[valid_mask],
                        'cell_ids': final_cluster_cells,
                        'coords': medoids_vox[final_cluster_cells] 
                    }
                    
        except Exception as e:
            print(f"Error loading {category} for {fish_id}: {e}")
            import traceback 
            print(f"\nCRITICAL CRASH inside {category} for fish {expt_ID}:")
            print(f"Error Type: {str(e)}")
            traceback.print_exc()
            continue
            
    return fish_id, fish_results


with ThreadPoolExecutor(max_workers=3) as executor:
    results = executor.map(process_fish_analysis, EXPT_FISH_LIST)
    for result in results:
        if result is not None:
            fish_id, fish_results = result
            loaded_cluster_data[fish_id] = fish_results

print("Data loading completed successfully")


# USE THIS
if not os.path.exists(TEMPLATE_BRAIN_PATH): 
    raise FileNotFoundError(f"Missing mandatory anatomical canvas template: {TEMPLATE_BRAIN_PATH}") 

nii_obj = nib.load(TEMPLATE_BRAIN_PATH) 
template_img = nii_obj.get_fdata() 

bg_canvas = np.max(template_img, axis=2).T 

plt.ioff() 

for expt_ID in EXPT_FISH_LIST: 
    print(f"\nEvaluating raw responder maps for: {expt_ID}") 
    fish_dir = BASE_DIR / PROJ_ID / expt_ID 
    vox_path = fish_dir / "medoids_template_vox.npy" 
    
    if not vox_path.exists(): 
        print(f" Warning: {vox_path} missing. Skipping this fish.") 
        continue 
        
    medoids_vox = np.load(vox_path) 
    valid_registration_mask = (medoids_vox >= 0).all(axis=1) 
    
    fig, axes = plt.subplots(2, 2, figsize=(14, 14)) 
    plt.suptitle(f"Pre-Clustering Whole Responder Distributions\n{expt_ID} (Top {100-PERCENTILE_CUTOFF}% Cutoff)", fontsize=15, y=0.98)
    axes = axes.flatten() 
    
    for idx, category in enumerate(RESPONSE_TYPES): 
        ax = axes[idx] 
        

        cluster_dir = fish_dir / f'FA_agglo_clustering_pct={PERCENTILE_CUTOFF}_c={N_CLUSTERS}'
        pipeline_idxs = [] 
        for c_idx in range(1, N_CLUSTERS + 1): 
            idx_file = cluster_dir / f"{category}_c{c_idx}_idxs.npy" 
            if idx_file.exists(): 
                pipeline_idxs.append(np.load(idx_file)) 
                
        if len(pipeline_idxs) == 0: 
            ax.text(0.5, 0.5, f"Missing cluster files for:\n{category}", ha='center', va='center', color='gray') 
            ax.axis("off") 
            continue 
            
        raw_responder_idxs = np.concatenate(pipeline_idxs) 
        valid_raw_idxs = raw_responder_idxs[valid_registration_mask[raw_responder_idxs]] 
        total_cells_found = len(valid_raw_idxs) 
        
        ax.imshow(bg_canvas, cmap="gray", origin="upper") 
        
        if total_cells_found == 0: 
            ax.text(0.5, 0.5, f"No Responding Cells\n(n = 0)", ha='center', va='center', color='gray') 
            ax.set_title(LABEL_TITLES[category], fontsize=12) 
            ax.axis("off") 
            continue 
            
        coords = medoids_vox[valid_raw_idxs] 
    
        ax.scatter( 
            coords[:, 0], 
            coords[:, 1], 
            color=CATEGORY_COLORS[category], 
            s=1.2, 
            alpha=0.35, 
            edgecolors='none' 
        ) 
        
        ax.set_title(f"{LABEL_TITLES[category]} Responder Pool (n = {total_cells_found} cells)", fontsize=13) 
        ax.set_xlim(0, bg_canvas.shape[1]) 
     
        ax.set_ylim(bg_canvas.shape[0], 0) 
        ax.axis("off") 
        
    fig.subplots_adjust(wspace=0.3, hspace=0.3) 
    fish_summary_dir = os.path.join(SUMMARY_PLOTS_DIR, expt_ID) 
    os.makedirs(fish_summary_dir, exist_ok=True) 
    
    output_path = os.path.join(fish_summary_dir, f"whole_distribution_plot_pct_{PERCENTILE_CUTOFF}.png")
    fig.savefig(output_path, dpi=200, bbox_inches='tight', pad_inches=0.4) 
    print(f" Successfully saved unified raw pool grid -> {output_path}") 
    plt.close(fig) 
    gc.collect() 

plt.ion()
TEMPLATE_BRAIN_PATH = "/mnt/storage-raid10/Yun/analysis_output/registration/template_mean_brain.nii.gz"
SUMMARY_PLOTS_DIR = "/ssd-pool/james/lightsheet/Zebrafish-whole-brain-analysis/unsupervised_plots_summary"
COLOR_PALETTE = plt.cm.get_cmap('tab10', N_CLUSTERS)

if not os.path.exists(TEMPLATE_BRAIN_PATH):
    raise FileNotFoundError(f"Missing mandatory anatomical canvas template: {TEMPLATE_BRAIN_PATH}")

nii_obj = nib.load(TEMPLATE_BRAIN_PATH)
template_img = nii_obj.get_fdata()

bg_canvas = np.max(template_img, axis=2).T 
plt.ioff()

for expt_ID in EXPT_FISH_LIST:
    if expt_ID not in loaded_cluster_data:
        continue
        
    print(f"\nProjecting anatomical spatial profiles for: {expt_ID}")
    fish_dir = BASE_DIR / PROJ_ID / expt_ID
    
    vox_path = fish_dir / "medoids_template_vox.npy"
    if not vox_path.exists():
        print(f"  Missing template voxels for {expt_ID}. Skipping.")
        continue
        
    medoids_vox = np.load(vox_path)
    valid_registration_mask = (medoids_vox >= 0).all(axis=1)

    cluster_dir = fish_dir / f'FA_agglo_clustering_pct={PERCENTILE_CUTOFF}_c={N_CLUSTERS}'
    
    for category in RESPONSE_TYPES:
        target_subfolder = os.path.join(SUMMARY_PLOTS_DIR, expt_ID, category)
        os.makedirs(target_subfolder, exist_ok=True)
        
        valid_cluster_files = []
        for c_idx in range(1, N_CLUSTERS + 1):
            idx_file = cluster_dir / f"{category}_c{c_idx}_idxs.npy"
            if idx_file.exists():
                cluster_indices = np.load(idx_file)
                if len(cluster_indices) > 0:
                    valid_cluster_indices = cluster_indices[valid_registration_mask[cluster_indices]]
                    if len(valid_cluster_indices) > 0:
                        valid_cluster_files.append((c_idx, idx_file, valid_cluster_indices))
                        
        n_valid_panels = len(valid_cluster_files)
        if n_valid_panels == 0:
            continue
     
        n_cols = min(6, n_valid_panels)
        n_rows = int(np.ceil(n_valid_panels / n_cols))
        
        fig, axes = plt.subplots(n_rows, n_cols, figsize=(6 * n_cols, 6 * n_rows), squeeze=False)
        plt.suptitle(f"{expt_ID} - {category.upper()} Spatial Cluster Maps (Top {100-PERCENTILE_CUTOFF}% Cutoff)", fontsize=15, y=0.98)
        axes = axes.flatten()
        
        plotted_panels = 0
        
        for c_idx, idx_file, valid_cluster_indices in valid_cluster_files:
            if plotted_panels >= len(axes):
                print(f"Omitted extra cluster panel {c_idx} to avoid array out-of-bounds")
                break
                
            coords = medoids_vox[valid_cluster_indices]
            ax = axes[plotted_panels]
  
         
            ax.imshow(bg_canvas, cmap="gray", origin="upper")
 
            ax.scatter(
                coords[:, 0], 
                coords[:, 1], 
                color=COLOR_PALETTE(c_idx - 1),
                s=2.5, 
                alpha=0.6, 
                edgecolors='none'
            )
            
            ax.set_title(f"Cluster {c_idx} (n={len(valid_cluster_indices)} cells)", fontsize=11)
            ax.set_xlim(0, bg_canvas.shape[1])

            ax.set_ylim(bg_canvas.shape[0], 0) 
            ax.axis("off")
            
            ax.text(0.05, 0.08, f"C{c_idx}", 
                    transform=ax.transAxes, 
                    color="white", 
                    fontsize=20, 
                    weight="bold", 
                    ha="left", 
                    va="bottom")
            plotted_panels += 1
            
        for empty_idx in range(plotted_panels, len(axes)):
            fig.delaxes(axes[empty_idx])
            
        if plotted_panels > 0:
            fig.subplots_adjust(wspace=0.1, hspace=0.1)
            output_path = os.path.join(target_subfolder, f"spatial_distribution_plot_pct_{PERCENTILE_CUTOFF}.png")
            fig.savefig(output_path, dpi=200, bbox_inches='tight', pad_inches=0.5)
            print(f"Saved plot for: {output_path}")
            
        plt.close(fig)
        
    gc.collect()

plt.ion()
print("\nAll corrected plots rendered successfully.")

In [1]:
import sys
import pandas as pd
import joblib
from tqdm import tqdm
from pathlib import Path
from sklearn.decomposition import TruncatedSVD

import os
import gc
from pathlib import Path
import numpy as np
import joblib
from scipy.stats import zscore
import matplotlib.pyplot as plt
import nibabel as nib
from sklearn.decomposition import FactorAnalysis
from sklearn.cluster import AgglomerativeClustering
from concurrent.futures import ProcessPoolExecutor 
import ants
from sklearn.decomposition import FactorAnalysis

from concurrent.futures import ThreadPoolExecutor
from mpl_toolkits.axes_grid1.inset_locator import inset_axes

BASE_DIR = Path("/mnt/storage-raid10/Yun/analysis_output/chemogenetic")
PROJ_ID = "hcrt-trpv1_huc-h2b-g8m_csn_120min"
PROJ_CTRL = "huc-h2b-g8m_csn_120min"
TEMPLATE_BRAIN_PATH = "/mnt/storage-raid10/Yun/analysis_output/registration/template_mean_brain.nii.gz"
SUMMARY_PLOTS_DIR = "/ssd-pool/james/lightsheet/Zebrafish-whole-brain-analysis/unsupervised_plots_summary"
stim_start_post = 2700
stim_end_post = 7200
PERCENTILE_CUTOFF = 90

EXPT_FISH_LIST = [
    #"251008_hcrt-trpv1_huc-h2b-g8m_csn_10uM_fish4",
    #"251102_hcrt-trpv1_huc-h2b-g8m_csn_10uM_fish1",
    #"251102_hcrt-trpv1_huc-h2b-g8m_csn_10uM_fish2",
    #"251210_hcrt-trpv1_huc-h2b-g8m_csn_10uM_fish1",
    #"251210_hcrt-trpv1_huc-h2b-g8m_csn_10uM_fish2", 
    #"251210_hcrt-trpv1_huc-h2b-g8m_csn_10uM_fish3",
    #"260514_hcrt-trpv1_huc-h2b-g8m_csn_10uM_fish1",
    #"260514_hcrt-trpv1_huc-h2b-g8m_csn_10uM_fish2",
    "260515_hcrt-trpv1_huc-h2b-g8m_csn_10uM_fish1"
]



RESPONSE_TYPES = [
    "tonic_pos",
    "tonic_neg",
    "phasic_pos",
    "phasic_neg"
                 ]

DIR_ANTS_OUTPUT = str(BASE_DIR)


N_COMPONENTS = 15
N_CLUSTERS = 7

CATEGORY_COLORS = {'tonic_pos': 'orange', 'tonic_neg': 'blue', 'phasic_pos': 'red', 'phasic_neg': 'purple'} 
LABEL_TITLES = {'tonic_pos': 'Tonic(+)', 'tonic_neg': 'Tonic(-)', 'phasic_pos': 'Phasic(+)', 'phasic_neg': 'Phasic(-)'} 

from multiprocessing import Pool, cpu_count

def process_single_experiment(args):
    expt_ID, PERCENTILE_CUTOFF, PROJ_ID, DIR_ANTS_OUTPUT, medoid_types, stim_start, stim_end, use_qc, t, save_dendrogram, max_cells, n_components = args
    
    dir_expt = os.path.join(DIR_ANTS_OUTPUT, PROJ_ID, expt_ID)
    OUTPUT_DIR = os.path.join(dir_expt, f"FA_agglo_clustering_pct={PERCENTILE_CUTOFF}_c={t}")
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    
    loaded_traces = {}

    for label in medoid_types:
        is_tonic = "tonic" in label
        data_filename = "f_tonic.npy" if is_tonic else "f_phasic.npy"
        data_path = os.path.join(dir_expt, data_filename)
        
        if not os.path.exists(data_path):
            continue

        if data_filename not in loaded_traces:
            loaded_traces[data_filename] = np.load(data_path)
        traces = loaded_traces[data_filename]

        
        if is_tonic:
            tonic_score = np.var(traces[:, stim_start:stim_end], axis=1)
            
            if 'pos' in label:
                tonic_cutoff = np.percentile(tonic_score, PERCENTILE_CUTOFF)
                idxs = np.where(tonic_score >= tonic_cutoff)[0]
            else:
                tonic_cutoff = np.percentile(tonic_score, 100 - PERCENTILE_CUTOFF)
                idxs = np.where(tonic_score <= tonic_cutoff)[0]
            
        else:
            dprime_path = os.path.join(dir_expt, "phasic_dprime_cells_raw.npy")
            if not os.path.exists(dprime_path):
                continue
            dprime = np.load(dprime_path)
            
            
            if "pos" in label:
                top_percentile_cutoff = np.nanpercentile(dprime, PERCENTILE_CUTOFF)
          
                idxs = np.where(dprime >= top_percentile_cutoff)[0]
            else:
                top_percentile_cutoff = np.nanpercentile(dprime, 100 - PERCENTILE_CUTOFF)
        
                idxs = np.where(dprime <= top_percentile_cutoff)[0]

          
            print(f"\n[{label} COMPUTE CHECK]")
            print(f"Top percentile value calculated: {top_percentile_cutoff}")
            print(f"Extracted array length before check: {len(idxs)}")
            
            idxs = idxs[idxs < traces.shape[0]]

            print(f" [{label}] Cells after bounds check: {len(idxs)}")

        if len(idxs) == 0:
            print(f" Warning: [{expt_ID} - {label}] No cells passed the percentile threshold.")
            continue
            
        trace_subset = traces[idxs, stim_start:stim_end]

        row_variances = np.var(trace_subset, axis=1)
        nonzero_variance_mask = row_variances > 1e-8
        
        if np.sum(nonzero_variance_mask) < 4:
            print(f" Warning: [{expt_ID} - {label}] Dropped. Only {np.sum(nonzero_variance_mask)} cells had non-zero variance during the stim window.")
            continue
            
      
        trace_subset = trace_subset[nonzero_variance_mask]
        idxs = idxs[nonzero_variance_mask]
        
        z_traces = zscore(trace_subset, axis=1)
        valid_mask = np.all(np.isfinite(z_traces), axis=1)
        
        if np.sum(valid_mask) < 4:
            print(f" Warning: [{expt_ID} - {label}] Dropped. Z-scoring still resulted in non-finite values.")
            continue
            
        z_traces = z_traces[valid_mask]
        idxs = idxs[valid_mask]

        MAX_TARGET_CELLS = 50000
        if len(idxs) > MAX_TARGET_CELLS:
    
            metric_arr = tonic_score[idxs] if is_tonic else dprime[idxs]
            
            if "pos" in label:
                top_orders = np.argsort(metric_arr)[-MAX_TARGET_CELLS:]
            else:
                top_orders = np.argsort(metric_arr)[:MAX_TARGET_CELLS]
        
            z_traces = z_traces[top_orders]
            idxs = idxs[top_orders]



        print(f"[{expt_ID} - {label}] Processing FA + Clustering with {z_traces.shape[0]} cells...")

        try:
            fa = FactorAnalysis(n_components=n_components, random_state=0)
            latent = fa.fit_transform(z_traces)
            fa_model_path = os.path.join(OUTPUT_DIR, f"FA_model_{label}.joblib")
            joblib.dump(fa, fa_model_path)

            clusterer = AgglomerativeClustering(n_clusters=t, linkage='ward')
            cluster_labels = clusterer.fit_predict(latent)

            for c_idx in range(1, t + 1):

                c_mask = (cluster_labels == (c_idx - 1))
                cluster_cell_ids = idxs[c_mask]
                out_idx_path = os.path.join(OUTPUT_DIR, f"{label}_c{c_idx}_idxs.npy")
                np.save(out_idx_path, cluster_cell_ids)

            del z_traces, latent, trace_subset, idxs
        except Exception as e:
            print(f"Failure processing cluster execution line for {label}: {e}")
            continue
        
    del loaded_traces
    gc.collect()
    return expt_ID

def hierarchical_clustering_parallel(
    EXPT_FISH_LIST, PERCENTILE_CUTOFF, PROJ_ID, DIR_ANTS_OUTPUT, RESPONSE_TYPES, 
    stim_start, stim_end, use_qc, t, save_dendrogram, max_cells, n_components
):
    tasks = [(
        expt_ID, PERCENTILE_CUTOFF, PROJ_ID, DIR_ANTS_OUTPUT, RESPONSE_TYPES, 
        stim_start, stim_end, use_qc, t, save_dendrogram, max_cells, n_components
    ) for expt_ID in EXPT_FISH_LIST]
    
    num_workers = min(4, max(1, int(cpu_count() * 0.75)))
    print(f"Starting parallel execution with {num_workers} workers.")
    
    with Pool(processes=num_workers) as pool:
        results = list(tqdm(pool.imap_unordered(process_single_experiment, tasks), total=len(tasks), desc="Parallel clustering"))        
    return results

if __name__ == "__main__":
    print("STEP 1: Running Parallel Agglomerative Clustering")

    hierarchical_clustering_parallel(
        EXPT_FISH_LIST=EXPT_FISH_LIST, PERCENTILE_CUTOFF=PERCENTILE_CUTOFF, PROJ_ID=PROJ_ID,
        DIR_ANTS_OUTPUT=DIR_ANTS_OUTPUT, RESPONSE_TYPES=RESPONSE_TYPES,
        stim_start=2700, stim_end=7200, use_qc = False,  t=N_CLUSTERS, n_components=N_COMPONENTS, 
        save_dendrogram=False,
        max_cells=50000
    )

loaded_cluster_data = {}
def process_fish_analysis(fish_id):
    fish_dir = BASE_DIR / PROJ_ID / fish_id

    cluster_dir = fish_dir / f'FA_agglo_clustering_pct={PERCENTILE_CUTOFF}_c={N_CLUSTERS}'
        
    if not cluster_dir.exists():
        print(f"Skipping {fish_id} — Clustering output directory not found at {cluster_dir}")
        return None
    vox_path = fish_dir / "medoids_template_vox.npy"
    if not vox_path.exists():
        print(f"Skipping {fish_id} — Coordinates array missing.")
        return None
    medoids_vox = np.load(vox_path)
    
    fish_results = {cat: {} for cat in RESPONSE_TYPES}
    
    for category in RESPONSE_TYPES:
        is_tonic = "tonic" in category
        data_filename = "f_tonic.npy" if is_tonic else "f_phasic.npy"
        data_path = fish_dir / data_filename
        
        if not data_path.exists():
            print(f'{data_path} does not exist')
            continue
            
        try:
            all_cluster_cells = []
            cluster_cell_mappings = {}
            
            for c_idx in range(1, N_CLUSTERS + 1):
                idx_file = cluster_dir / f"{category}_c{c_idx}_idxs.npy"
                if idx_file.exists():
                    c_cells = np.load(idx_file)
                    if len(c_cells) > 0:
                        all_cluster_cells.append(c_cells)
                        cluster_cell_mappings[c_idx] = c_cells
                        
            if not all_cluster_cells:
                continue
                
            unique_responders = np.unique(np.concatenate(all_cluster_cells))
                
            raw_traces_mmap = np.load(data_path, mmap_mode="r")
            pooled_post_traces = np.array(raw_traces_mmap[unique_responders, stim_start_post:stim_end_post])
            pooled_full_traces = np.array(raw_traces_mmap[unique_responders, 0:7200])
            cell_to_ram_idx = {cell: i for i, cell in enumerate(unique_responders)}
            
            for c_idx in range(1, N_CLUSTERS + 1):
                if c_idx not in cluster_cell_mappings:
                    continue
                    
                cluster_cells = cluster_cell_mappings[c_idx]
                ram_indices = [cell_to_ram_idx[cell] for cell in cluster_cells]
                
                c_post = pooled_post_traces[ram_indices]
                c_full = pooled_full_traces[ram_indices]
                
                z_post = zscore(c_post, axis=1)
                z_full = zscore(c_full, axis=1)
                
                valid_mask = np.all(np.isfinite(z_post), axis=1) & np.all(np.isfinite(z_full), axis=1)
                
                if np.sum(valid_mask) > 0:
            
                    final_cluster_cells = cluster_cells[valid_mask]
                    
                    fish_results[category][c_idx] = {
                        'post_drug': z_post[valid_mask],
                        'full_timeline': z_full[valid_mask],
                        'cell_ids': final_cluster_cells,
                        'coords': medoids_vox[final_cluster_cells] 
                    }
                    
        except Exception as e:
            print(f"Error loading {category} for {fish_id}: {e}")
            import traceback 
            print(f"\nCRITICAL CRASH inside {category} for fish {expt_ID}:")
            print(f"Error Type: {str(e)}")
            traceback.print_exc()
            continue
            
    return fish_id, fish_results


with ThreadPoolExecutor(max_workers=3) as executor:
    results = executor.map(process_fish_analysis, EXPT_FISH_LIST)
    for result in results:
        if result is not None:
            fish_id, fish_results = result
            loaded_cluster_data[fish_id] = fish_results

print("Data loading completed successfully")


# USE THIS
if not os.path.exists(TEMPLATE_BRAIN_PATH): 
    raise FileNotFoundError(f"Missing mandatory anatomical canvas template: {TEMPLATE_BRAIN_PATH}") 

nii_obj = nib.load(TEMPLATE_BRAIN_PATH) 
template_img = nii_obj.get_fdata() 

bg_canvas = np.max(template_img, axis=2).T 

plt.ioff() 

for expt_ID in EXPT_FISH_LIST: 
    print(f"\nEvaluating raw responder maps for: {expt_ID}") 
    fish_dir = BASE_DIR / PROJ_ID / expt_ID 
    vox_path = fish_dir / "medoids_template_vox.npy" 
    
    if not vox_path.exists(): 
        print(f" Warning: {vox_path} missing. Skipping this fish.") 
        continue 
        
    medoids_vox = np.load(vox_path) 
    valid_registration_mask = (medoids_vox >= 0).all(axis=1) 
    
    fig, axes = plt.subplots(2, 2, figsize=(14, 14)) 
    plt.suptitle(f"Pre-Clustering Whole Responder Distributions\n{expt_ID} (Top {100-PERCENTILE_CUTOFF}% Cutoff)", fontsize=15, y=0.98)
    axes = axes.flatten() 
    
    for idx, category in enumerate(RESPONSE_TYPES): 
        ax = axes[idx] 
        

        cluster_dir = fish_dir / f'FA_agglo_clustering_pct={PERCENTILE_CUTOFF}_c={N_CLUSTERS}'
        pipeline_idxs = [] 
        for c_idx in range(1, N_CLUSTERS + 1): 
            idx_file = cluster_dir / f"{category}_c{c_idx}_idxs.npy" 
            if idx_file.exists(): 
                pipeline_idxs.append(np.load(idx_file)) 
                
        if len(pipeline_idxs) == 0: 
            ax.text(0.5, 0.5, f"Missing cluster files for:\n{category}", ha='center', va='center', color='gray') 
            ax.axis("off") 
            continue 
            
        raw_responder_idxs = np.concatenate(pipeline_idxs) 
        valid_raw_idxs = raw_responder_idxs[valid_registration_mask[raw_responder_idxs]] 
        total_cells_found = len(valid_raw_idxs) 
        
        ax.imshow(bg_canvas, cmap="gray", origin="upper") 
        
        if total_cells_found == 0: 
            ax.text(0.5, 0.5, f"No Responding Cells\n(n = 0)", ha='center', va='center', color='gray') 
            ax.set_title(LABEL_TITLES[category], fontsize=12) 
            ax.axis("off") 
            continue 
            
        coords = medoids_vox[valid_raw_idxs] 
    
        ax.scatter( 
            coords[:, 0], 
            coords[:, 1], 
            color=CATEGORY_COLORS[category], 
            s=1.2, 
            alpha=0.35, 
            edgecolors='none' 
        ) 
        
        ax.set_title(f"{LABEL_TITLES[category]} Responder Pool (n = {total_cells_found} cells)", fontsize=13) 
        ax.set_xlim(0, bg_canvas.shape[1]) 
     
        ax.set_ylim(bg_canvas.shape[0], 0) 
        ax.axis("off") 
        
    fig.subplots_adjust(wspace=0.3, hspace=0.3) 
    fish_summary_dir = os.path.join(SUMMARY_PLOTS_DIR, expt_ID) 
    os.makedirs(fish_summary_dir, exist_ok=True) 
    
    output_path = os.path.join(fish_summary_dir, f"whole_distribution_plot_pct_{PERCENTILE_CUTOFF}.png")
    fig.savefig(output_path, dpi=200, bbox_inches='tight', pad_inches=0.4) 
    print(f" Successfully saved unified raw pool grid -> {output_path}") 
    plt.close(fig) 
    gc.collect() 

plt.ion()

#this one - plot spatila dsteibution into clusters
TEMPLATE_BRAIN_PATH = "/mnt/storage-raid10/Yun/analysis_output/registration/template_mean_brain.nii.gz"
SUMMARY_PLOTS_DIR = "/ssd-pool/james/lightsheet/Zebrafish-whole-brain-analysis/unsupervised_plots_summary"
COLOR_PALETTE = plt.cm.get_cmap('tab10', N_CLUSTERS)

if not os.path.exists(TEMPLATE_BRAIN_PATH):
    raise FileNotFoundError(f"Missing mandatory anatomical canvas template: {TEMPLATE_BRAIN_PATH}")

nii_obj = nib.load(TEMPLATE_BRAIN_PATH)
template_img = nii_obj.get_fdata()

bg_canvas = np.max(template_img, axis=2).T 
plt.ioff()

for expt_ID in EXPT_FISH_LIST:
    if expt_ID not in loaded_cluster_data:
        continue
        
    print(f"\nProjecting anatomical spatial profiles for: {expt_ID}")
    fish_dir = BASE_DIR / PROJ_ID / expt_ID
    
    vox_path = fish_dir / "medoids_template_vox.npy"
    if not vox_path.exists():
        print(f"  Missing template voxels for {expt_ID}. Skipping.")
        continue
        
    medoids_vox = np.load(vox_path)
    valid_registration_mask = (medoids_vox >= 0).all(axis=1)

    cluster_dir = fish_dir / f'FA_agglo_clustering_pct={PERCENTILE_CUTOFF}_c={N_CLUSTERS}'
    
    for category in RESPONSE_TYPES:
        target_subfolder = os.path.join(SUMMARY_PLOTS_DIR, expt_ID, category)
        os.makedirs(target_subfolder, exist_ok=True)
        
        valid_cluster_files = []
        for c_idx in range(1, N_CLUSTERS + 1):
            idx_file = cluster_dir / f"{category}_c{c_idx}_idxs.npy"
            if idx_file.exists():
                cluster_indices = np.load(idx_file)
                if len(cluster_indices) > 0:
                    valid_cluster_indices = cluster_indices[valid_registration_mask[cluster_indices]]
                    if len(valid_cluster_indices) > 0:
                        valid_cluster_files.append((c_idx, idx_file, valid_cluster_indices))
                        
        n_valid_panels = len(valid_cluster_files)
        if n_valid_panels == 0:
            continue
     
        n_cols = min(6, n_valid_panels)
        n_rows = int(np.ceil(n_valid_panels / n_cols))
        
        fig, axes = plt.subplots(n_rows, n_cols, figsize=(6 * n_cols, 6 * n_rows), squeeze=False)
        plt.suptitle(f"{expt_ID} - {category.upper()} Spatial Cluster Maps (Top {100-PERCENTILE_CUTOFF}% Cutoff)", fontsize=15, y=0.98)
        axes = axes.flatten()
        
        plotted_panels = 0
        
        for c_idx, idx_file, valid_cluster_indices in valid_cluster_files:
            if plotted_panels >= len(axes):
                print(f"Omitted extra cluster panel {c_idx} to avoid array out-of-bounds")
                break
                
            coords = medoids_vox[valid_cluster_indices]
            ax = axes[plotted_panels]
  
         
            ax.imshow(bg_canvas, cmap="gray", origin="upper")
 
            ax.scatter(
                coords[:, 0], 
                coords[:, 1], 
                color=COLOR_PALETTE(c_idx - 1),
                s=2.5, 
                alpha=0.6, 
                edgecolors='none'
            )
            
            ax.set_title(f"Cluster {c_idx} (n={len(valid_cluster_indices)} cells)", fontsize=11)
            ax.set_xlim(0, bg_canvas.shape[1])

            ax.set_ylim(bg_canvas.shape[0], 0) 
            ax.axis("off")
            
            ax.text(0.05, 0.08, f"C{c_idx}", 
                    transform=ax.transAxes, 
                    color="white", 
                    fontsize=20, 
                    weight="bold", 
                    ha="left", 
                    va="bottom")
            plotted_panels += 1
            
        for empty_idx in range(plotted_panels, len(axes)):
            fig.delaxes(axes[empty_idx])
            
        if plotted_panels > 0:
            fig.subplots_adjust(wspace=0.1, hspace=0.1)
            output_path = os.path.join(target_subfolder, f"spatial_distribution_plot_pct_{PERCENTILE_CUTOFF}.png")
            fig.savefig(output_path, dpi=200, bbox_inches='tight', pad_inches=0.5)
            print(f"Saved plot for: {output_path}")
            
        plt.close(fig)
        
    gc.collect()

plt.ion()
print("\nAll corrected plots rendered successfully.")

STEP 1: Running Parallel Agglomerative Clustering
Starting parallel execution with 4 workers.


Parallel clustering:   0%|          | 0/1 [00:00<?, ?it/s]

[260515_hcrt-trpv1_huc-h2b-g8m_csn_10uM_fish1 - tonic_pos] Processing FA + Clustering with 50000 cells...
[260515_hcrt-trpv1_huc-h2b-g8m_csn_10uM_fish1 - tonic_neg] Processing FA + Clustering with 50000 cells...

[phasic_pos COMPUTE CHECK]
Top percentile value calculated: 0.14406009018421173
Extracted array length before check: 53101
 [phasic_pos] Cells after bounds check: 53101
[260515_hcrt-trpv1_huc-h2b-g8m_csn_10uM_fish1 - phasic_pos] Processing FA + Clustering with 50000 cells...

[phasic_neg COMPUTE CHECK]
Top percentile value calculated: -0.074953094124794
Extracted array length before check: 53101
 [phasic_neg] Cells after bounds check: 53101
[260515_hcrt-trpv1_huc-h2b-g8m_csn_10uM_fish1 - phasic_neg] Processing FA + Clustering with 50000 cells...


Parallel clustering: 100%|██████████| 1/1 [23:10<00:00, 1390.35s/it]


Data loading completed successfully

Evaluating raw responder maps for: 260515_hcrt-trpv1_huc-h2b-g8m_csn_10uM_fish1
 Successfully saved unified raw pool grid -> /ssd-pool/james/lightsheet/Zebrafish-whole-brain-analysis/unsupervised_plots_summary/260515_hcrt-trpv1_huc-h2b-g8m_csn_10uM_fish1/whole_distribution_plot_pct_90.png


/tmp/ipykernel_249817/314036795.py:401: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  COLOR_PALETTE = plt.cm.get_cmap('tab10', N_CLUSTERS)



Projecting anatomical spatial profiles for: 260515_hcrt-trpv1_huc-h2b-g8m_csn_10uM_fish1
Saved plot for: /ssd-pool/james/lightsheet/Zebrafish-whole-brain-analysis/unsupervised_plots_summary/260515_hcrt-trpv1_huc-h2b-g8m_csn_10uM_fish1/tonic_pos/spatial_distribution_plot_pct_90.png
Saved plot for: /ssd-pool/james/lightsheet/Zebrafish-whole-brain-analysis/unsupervised_plots_summary/260515_hcrt-trpv1_huc-h2b-g8m_csn_10uM_fish1/tonic_neg/spatial_distribution_plot_pct_90.png
Saved plot for: /ssd-pool/james/lightsheet/Zebrafish-whole-brain-analysis/unsupervised_plots_summary/260515_hcrt-trpv1_huc-h2b-g8m_csn_10uM_fish1/phasic_pos/spatial_distribution_plot_pct_90.png
Saved plot for: /ssd-pool/james/lightsheet/Zebrafish-whole-brain-analysis/unsupervised_plots_summary/260515_hcrt-trpv1_huc-h2b-g8m_csn_10uM_fish1/phasic_neg/spatial_distribution_plot_pct_90.png

All corrected plots rendered successfully.
